In [ ]:
from pathlib import Path
import json
import re
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, average_precision_score, balanced_accuracy_score, confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import StratifiedGroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder, StandardScaler


output_dir = Path("/kaggle/working/results/phase1")
output_dir.mkdir(parents=True, exist_ok=True)

data_sources = {
    "nsl_kdd": [
        "/kaggle/input/nsl-kdd/KDDTrain+.txt",
        "/kaggle/input/nsl-kdd/KDDTest+.txt"
    ],
    "cicids2017": [
        "/kaggle/input/cicids2017"
    ],
    "unsw_nb15": [
        "/kaggle/input/unsw-nb15"
    ]
}

split_config = {
    "train_fraction": 0.70,
    "valid_fraction": 0.15,
    "test_fraction": 0.15,
    "random_state": 42
}

nsl_columns = [
    "duration", "protocol_type", "service", "flag", "src_bytes", "dst_bytes", "land",
    "wrong_fragment", "urgent", "hot", "num_failed_logins", "logged_in", "num_compromised",
    "root_shell", "su_attempted", "num_root", "num_file_creations", "num_shells",
    "num_access_files", "num_outbound_cmds", "is_host_login", "is_guest_login", "count",
    "srv_count", "serror_rate", "srv_serror_rate", "rerror_rate", "srv_rerror_rate",
    "same_srv_rate", "diff_srv_rate", "srv_diff_host_rate", "dst_host_count",
    "dst_host_srv_count", "dst_host_same_srv_rate", "dst_host_diff_srv_rate",
    "dst_host_same_src_port_rate", "dst_host_srv_diff_host_rate", "dst_host_serror_rate",
    "dst_host_srv_serror_rate", "dst_host_rerror_rate", "dst_host_srv_rerror_rate",
    "raw_label", "difficulty"
]

nsl_family_map = {
    "normal": "normal",
    "back": "dos",
    "land": "dos",
    "neptune": "dos",
    "pod": "dos",
    "smurf": "dos",
    "teardrop": "dos",
    "mailbomb": "dos",
    "apache2": "dos",
    "processtable": "dos",
    "udpstorm": "dos",
    "ipsweep": "probe",
    "nmap": "probe",
    "portsweep": "probe",
    "satan": "probe",
    "mscan": "probe",
    "saint": "probe",
    "ftp_write": "r2l",
    "guess_passwd": "r2l",
    "imap": "r2l",
    "multihop": "r2l",
    "phf": "r2l",
    "spy": "r2l",
    "warezclient": "r2l",
    "warezmaster": "r2l",
    "sendmail": "r2l",
    "named": "r2l",
    "snmpgetattack": "r2l",
    "snmpguess": "r2l",
    "xlock": "r2l",
    "xsnoop": "r2l",
    "worm": "r2l",
    "buffer_overflow": "u2r",
    "loadmodule": "u2r",
    "perl": "u2r",
    "rootkit": "u2r",
    "httptunnel": "u2r",
    "ps": "u2r",
    "sqlattack": "u2r",
    "xterm": "u2r"
}

attack_label_candidates = [
    "label", "labels", "class", "target", "attack", "attack_type", "attack_cat",
    "category", "traffic_label", "raw_label", "rawlabel"
]

family_label_candidates = [
    "attack_family", "family", "attack_cat", "attack_category", "category_group"
]

drop_candidates = {
    "flow_id", "flowid", "src ip", "dst ip", "srcip", "dstip", "timestamp", "time", "id",
    "index", "idx", "session_id", "record_id"
}


def normalize_name(value):
    value = str(value).strip().lower()
    value = re.sub(r"[\s\-/]+", "_", value)
    value = re.sub(r"[^a-z0-9_]", "", value)
    value = re.sub(r"_+", "_", value).strip("_")
    return value


def normalize_text(value):
    if pd.isna(value):
        return ""
    value = str(value).strip().lower()
    value = value.replace("\x00", "")
    value = re.sub(r"\s+", " ", value)
    return value


def resolve_files(entries):
    files = []
    for entry in entries:
        entry_path = Path(entry)
        if entry_path.is_dir():
            files.extend(sorted([p for p in entry_path.rglob("*") if p.suffix.lower() in {".csv", ".txt", ".data", ".parquet"}]))
        elif entry_path.exists():
            files.append(entry_path)
    return files


def read_single_file(path_obj):
    suffix = path_obj.suffix.lower()
    if suffix == ".parquet":
        frame = pd.read_parquet(path_obj)
        return frame
    try:
        frame = pd.read_csv(path_obj, low_memory=False)
        if frame.shape[1] > 1:
            return frame
    except Exception:
        pass
    try:
        frame = pd.read_csv(path_obj, header=None, low_memory=False)
        if frame.shape[1] > 1:
            return frame
    except Exception:
        pass
    try:
        frame = pd.read_csv(path_obj, sep=None, engine="python", low_memory=False)
        return frame
    except Exception:
        return pd.read_csv(path_obj, sep=r"\s+", engine="python", low_memory=False)


def standardize_columns(frame):
    cols = [normalize_name(c) for c in frame.columns]
    used = {}
    final_cols = []
    for col in cols:
        if col not in used:
            used[col] = 0
            final_cols.append(col)
        else:
            used[col] += 1
            final_cols.append(f"{col}_{used[col]}")
    frame.columns = final_cols
    return frame


def assign_known_schema(dataset_key, frame):
    if dataset_key == "nsl_kdd":
        if frame.shape[1] == len(nsl_columns):
            frame.columns = nsl_columns
        elif frame.shape[1] == len(nsl_columns) - 1:
            frame.columns = nsl_columns[:-1]
            frame["difficulty"] = 0
    return frame


def load_dataset(dataset_key, entries):
    parts = []
    for path_obj in resolve_files(entries):
        frame = read_single_file(path_obj)
        frame = assign_known_schema(dataset_key, frame)
        frame = standardize_columns(frame)
        frame["source_file"] = path_obj.name
        parts.append(frame)
    if not parts:
        raise FileNotFoundError(dataset_key)
    frame = pd.concat(parts, axis=0, ignore_index=True)
    frame = standardize_columns(frame)
    return frame


def select_first_present(columns, candidates):
    lookup = {normalize_name(c): c for c in columns}
    for candidate in candidates:
        key = normalize_name(candidate)
        if key in lookup:
            return lookup[key]
    return None


def derive_label_columns(dataset_key, frame):
    label_col = select_first_present(frame.columns, attack_label_candidates)
    family_col = select_first_present(frame.columns, family_label_candidates)

    if dataset_key == "nsl_kdd":
        if "raw_label" in frame.columns:
            label_col = "raw_label"
        family_col = None

    if dataset_key == "unsw_nb15":
        if "attack_cat" in frame.columns:
            family_col = "attack_cat"
        if "label" in frame.columns:
            label_col = "label"

    if dataset_key == "cicids2017":
        if "label" in frame.columns:
            label_col = "label"

    if label_col is None:
        raise ValueError(dataset_key)
    return label_col, family_col


def coerce_basic_types(frame):
    frame = frame.copy()
    frame = frame.replace([np.inf, -np.inf], np.nan)
    entirely_missing = [c for c in frame.columns if frame[c].isna().all()]
    if entirely_missing:
        frame = frame.drop(columns=entirely_missing)
    return frame


def drop_redundant_columns(frame, protected):
    columns_to_drop = []
    for col in frame.columns:
        if col in protected:
            continue
        if normalize_name(col) in drop_candidates:
            columns_to_drop.append(col)
            continue
        if frame[col].nunique(dropna=False) <= 1:
            columns_to_drop.append(col)
    if columns_to_drop:
        frame = frame.drop(columns=columns_to_drop)
    return frame


def nsl_family_from_label(raw_value):
    raw_value = normalize_text(raw_value).rstrip(".")
    return nsl_family_map.get(raw_value, "other_attack")


def cicids_family_from_label(raw_value):
    value = normalize_text(raw_value)
    if value in {"benign", "normal"}:
        return "normal"
    if "ddos" in value:
        return "ddos"
    if value.startswith("dos") or "hulk" in value or "goldeneye" in value or "slowhttptest" in value or "slowloris" in value:
        return "dos"
    if "portscan" in value or "port_scan" in value:
        return "portscan"
    if "bot" in value:
        return "botnet"
    if "web attack" in value or "sql injection" in value or "xss" in value or "brute force" in value:
        return "web_attack"
    if "ftp-patator" in value or "ssh-patator" in value:
        return "brute_force"
    if "infiltration" in value:
        return "infiltration"
    if "heartbleed" in value:
        return "heartbleed"
    return normalize_name(value) if value else "unknown"


def unsw_family_from_values(raw_label_value, family_value):
    family_text = normalize_text(family_value)
    if family_text:
        if family_text in {"normal", "benign"}:
            return "normal"
        return normalize_name(family_text)
    raw_text = normalize_text(raw_label_value)
    if raw_text in {"0", "normal", "benign"}:
        return "normal"
    if raw_text in {"1", "attack", "malicious"}:
        return "generic_attack"
    return "unknown"


def derive_family(dataset_key, raw_label_value, family_value=None):
    if dataset_key == "nsl_kdd":
        return nsl_family_from_label(raw_label_value)
    if dataset_key == "cicids2017":
        return cicids_family_from_label(raw_label_value)
    if dataset_key == "unsw_nb15":
        return unsw_family_from_values(raw_label_value, family_value)
    family_text = normalize_text(family_value)
    if family_text in {"normal", "benign"}:
        return "normal"
    if family_text:
        return normalize_name(family_text)
    raw_text = normalize_text(raw_label_value)
    if raw_text in {"0", "normal", "benign"}:
        return "normal"
    if raw_text in {"1", "attack", "malicious"}:
        return "generic_attack"
    return normalize_name(raw_text) if raw_text else "unknown"


def derive_binary(raw_label_value, family_value, dataset_key):
    family_name = derive_family(dataset_key, raw_label_value, family_value)
    return 0 if family_name == "normal" else 1


def stable_record_hash(frame, feature_columns):
    hash_frame = frame[feature_columns].copy()
    for col in hash_frame.columns:
        if pd.api.types.is_numeric_dtype(hash_frame[col]):
            hash_frame[col] = hash_frame[col].astype("float64").round(8)
        else:
            hash_frame[col] = hash_frame[col].astype(str).str.strip().str.lower()
    return pd.util.hash_pandas_object(hash_frame, index=False).astype(str)


def choose_feature_columns(frame, protected_columns):
    selected = [c for c in frame.columns if c not in protected_columns]
    return selected


def make_split_column(frame, stratify_col, group_col, config):
    splitter_outer = StratifiedGroupShuffleSplit(
        n_splits=1,
        test_size=config["test_fraction"],
        random_state=config["random_state"]
    )
    outer_train_idx, test_idx = next(splitter_outer.split(frame, frame[stratify_col], groups=frame[group_col]))

    remainder = frame.iloc[outer_train_idx].copy()
    remainder_y = remainder[stratify_col]
    remainder_groups = remainder[group_col]

    inner_valid_fraction = config["valid_fraction"] / (config["train_fraction"] + config["valid_fraction"])
    splitter_inner = StratifiedGroupShuffleSplit(
        n_splits=1,
        test_size=inner_valid_fraction,
        random_state=config["random_state"]
    )
    train_idx_local, valid_idx_local = next(splitter_inner.split(remainder, remainder_y, groups=remainder_groups))

    frame = frame.copy()
    frame["split_name"] = "train"
    frame.loc[frame.index[test_idx], "split_name"] = "test"
    frame.loc[remainder.index[valid_idx_local], "split_name"] = "valid"
    frame.loc[remainder.index[train_idx_local], "split_name"] = "train"
    return frame


def metric_pack(y_true, y_prob, threshold=0.5):
    y_pred = (y_prob >= threshold).astype(int)
    acc = accuracy_score(y_true, y_pred)
    pre = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    f1v = f1_score(y_true, y_pred, zero_division=0)
    bacc = balanced_accuracy_score(y_true, y_pred)
    try:
        auc = roc_auc_score(y_true, y_prob)
    except Exception:
        auc = np.nan
    try:
        ap = average_precision_score(y_true, y_prob)
    except Exception:
        ap = np.nan
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    fpr = fp / (fp + tn) if (fp + tn) > 0 else np.nan
    return {
        "accuracy": acc,
        "precision": pre,
        "recall": rec,
        "f1": f1v,
        "balanced_accuracy": bacc,
        "roc_auc": auc,
        "pr_auc": ap,
        "false_positive_rate": fpr,
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp)
    }


def build_pipeline(frame, feature_columns):
    numeric_columns = [c for c in feature_columns if pd.api.types.is_numeric_dtype(frame[c])]
    categorical_columns = [c for c in feature_columns if c not in numeric_columns]

    numeric_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])

    categorical_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1))
    ])

    transformer = ColumnTransformer(
        transformers=[
            ("num", numeric_pipe, numeric_columns),
            ("cat", categorical_pipe, categorical_columns)
        ],
        remainder="drop"
    )

    learner = HistGradientBoostingClassifier(
        learning_rate=0.05,
        max_depth=None,
        max_iter=300,
        min_samples_leaf=20,
        random_state=42
    )

    model = Pipeline([
        ("transformer", transformer),
        ("learner", learner)
    ])

    return model


def distribution_rows(frame, dataset_key):
    rows = []

    by_type = (
        frame.groupby(["raw_attack_label", "family_label", "binary_label", "split_name"], dropna=False)
        .size()
        .reset_index(name="count")
    )

    type_pivot = (
        by_type.pivot_table(
            index=["raw_attack_label", "family_label", "binary_label"],
            columns="split_name",
            values="count",
            fill_value=0
        )
        .reset_index()
    )

    for col in ["train", "valid", "test"]:
        if col not in type_pivot.columns:
            type_pivot[col] = 0

    type_pivot["overall"] = type_pivot[["train", "valid", "test"]].sum(axis=1)
    type_pivot["dataset_id"] = dataset_key
    type_pivot["level"] = "attack_type"
    rows.append(type_pivot[["dataset_id", "level", "raw_attack_label", "family_label", "binary_label", "overall", "train", "valid", "test"]])

    by_family = (
        frame.groupby(["family_label", "binary_label", "split_name"], dropna=False)
        .size()
        .reset_index(name="count")
    )

    family_pivot = (
        by_family.pivot_table(
            index=["family_label", "binary_label"],
            columns="split_name",
            values="count",
            fill_value=0
        )
        .reset_index()
    )

    for col in ["train", "valid", "test"]:
        if col not in family_pivot.columns:
            family_pivot[col] = 0

    family_pivot["overall"] = family_pivot[["train", "valid", "test"]].sum(axis=1)
    family_pivot["dataset_id"] = dataset_key
    family_pivot["level"] = "attack_family"
    family_pivot["raw_attack_label"] = ""
    rows.append(family_pivot[["dataset_id", "level", "raw_attack_label", "family_label", "binary_label", "overall", "train", "valid", "test"]])

    return pd.concat(rows, axis=0, ignore_index=True)


def summary_block(dataset_key, frame, feature_columns, label_col, family_col, duplicates_removed):
    total_count = len(frame)
    split_counts = frame["split_name"].value_counts().to_dict()
    family_counts = frame["family_label"].value_counts().sort_values(ascending=False).to_dict()
    binary_counts = frame["binary_label"].value_counts().sort_index().to_dict()
    missing_ratio = float(frame[feature_columns].isna().mean().mean()) if feature_columns else 0.0

    payload = {
        "dataset_id": dataset_key,
        "shape": list(frame.shape),
        "feature_count": len(feature_columns),
        "label_column": label_col,
        "family_column_source": family_col if family_col is not None else "",
        "duplicates_removed": int(duplicates_removed),
        "record_count": int(total_count),
        "split_counts": {k: int(v) for k, v in split_counts.items()},
        "binary_counts": {str(k): int(v) for k, v in binary_counts.items()},
        "family_counts": {str(k): int(v) for k, v in family_counts.items()},
        "average_missing_ratio_over_features": missing_ratio
    }

    text = []
    text.append(f"dataset_id: {payload['dataset_id']}")
    text.append(f"shape: {payload['shape']}")
    text.append(f"feature_count: {payload['feature_count']}")
    text.append(f"label_column: {payload['label_column']}")
    text.append(f"family_column_source: {payload['family_column_source']}")
    text.append(f"duplicates_removed: {payload['duplicates_removed']}")
    text.append(f"record_count: {payload['record_count']}")
    text.append(f"split_counts: {json.dumps(payload['split_counts'], ensure_ascii=False)}")
    text.append(f"binary_counts: {json.dumps(payload['binary_counts'], ensure_ascii=False)}")
    text.append(f"family_counts: {json.dumps(payload['family_counts'], ensure_ascii=False)}")
    text.append(f"average_missing_ratio_over_features: {payload['average_missing_ratio_over_features']:.8f}")
    return "\n".join(text)


metric_rows = []
protocol_rows = []
summary_texts = []

for dataset_key, entries in data_sources.items():
    raw_frame = load_dataset(dataset_key, entries)
    raw_frame = coerce_basic_types(raw_frame)
    label_col, family_col = derive_label_columns(dataset_key, raw_frame)

    work_frame = raw_frame.copy()
    work_frame["raw_attack_label"] = work_frame[label_col].astype(str).map(normalize_text)

    if family_col is not None and family_col in work_frame.columns:
        family_values = work_frame[family_col]
    else:
        family_values = pd.Series([""] * len(work_frame), index=work_frame.index)

    work_frame["family_label"] = [
        derive_family(dataset_key, raw_val, fam_val)
        for raw_val, fam_val in zip(work_frame["raw_attack_label"], family_values)
    ]
    work_frame["binary_label"] = (work_frame["family_label"] != "normal").astype(int)

    protected_before_drop = {"source_file", "raw_attack_label", "family_label", "binary_label"}
    work_frame = drop_redundant_columns(work_frame, protected_before_drop)

    protected_after_drop = {"source_file", "raw_attack_label", "family_label", "binary_label"}
    feature_columns = choose_feature_columns(work_frame, protected_after_drop)

    work_frame["record_hash"] = stable_record_hash(work_frame, feature_columns)
    duplicates_removed = int(work_frame.duplicated(subset=["record_hash"]).sum())
    work_frame = work_frame.drop_duplicates(subset=["record_hash"]).reset_index(drop=True)

    protected_after_hash = {"source_file", "raw_attack_label", "family_label", "binary_label", "record_hash"}
    feature_columns = choose_feature_columns(work_frame, protected_after_hash)

    stratify_source = work_frame["family_label"].copy()
    rare_families = stratify_source.value_counts()
    rare_families = set(rare_families[rare_families < 3].index.tolist())
    stratify_source = stratify_source.apply(lambda x: "rare_attack" if x in rare_families and x != "normal" else x)
    work_frame["stratify_label"] = stratify_source.astype(str)

    work_frame = make_split_column(work_frame, "stratify_label", "record_hash", split_config)

    train_frame = work_frame[work_frame["split_name"] == "train"].copy()
    valid_frame = work_frame[work_frame["split_name"] == "valid"].copy()
    test_frame = work_frame[work_frame["split_name"] == "test"].copy()

    model = build_pipeline(train_frame, feature_columns)
    model.fit(train_frame[feature_columns], train_frame["binary_label"])

    split_frames = {
        "train": train_frame,
        "valid": valid_frame,
        "test": test_frame
    }

    for split_name, split_frame in split_frames.items():
        probabilities = model.predict_proba(split_frame[feature_columns])[:, 1]
        metrics = metric_pack(split_frame["binary_label"].values, probabilities, threshold=0.5)
        metric_rows.append({
            "dataset_id": dataset_key,
            "split_name": split_name,
            "sample_count": int(len(split_frame)),
            "benign_count": int((split_frame["binary_label"] == 0).sum()),
            "attack_count": int((split_frame["binary_label"] == 1).sum()),
            **metrics
        })

    protocol_frame = distribution_rows(work_frame, dataset_key)
    protocol_frame["train_fraction"] = split_config["train_fraction"]
    protocol_frame["valid_fraction"] = split_config["valid_fraction"]
    protocol_frame["test_fraction"] = split_config["test_fraction"]
    protocol_frame["grouping_key"] = "record_hash"
    protocol_frame["stratification_key"] = "family_label_with_rare_attack_fallback"
    protocol_frame["duplicate_policy"] = "drop_exact_feature_hash_duplicates_before_split"
    protocol_rows.append(protocol_frame)

    summary_texts.append(summary_block(dataset_key, work_frame, feature_columns, label_col, family_col, duplicates_removed))

metrics_frame = pd.DataFrame(metric_rows).sort_values(["dataset_id", "split_name"]).reset_index(drop=True)
protocol_frame = pd.concat(protocol_rows, axis=0, ignore_index=True)

metrics_frame.to_csv(output_dir / "phase1_cross_dataset_metrics.csv", index=False)
protocol_frame.to_csv(output_dir / "phase1_attack_family_protocol.csv", index=False)

with open(output_dir / "phase1_dataset_summary.txt", "w", encoding="utf-8") as handle:
    handle.write("\n\n".join(summary_texts))

In [ ]:
from pathlib import Path
import copy
import json
import random
import re
import warnings

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shap
import torch
import torch.nn as nn
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, average_precision_score, balanced_accuracy_score, confusion_matrix, f1_score, precision_recall_fscore_support, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import StratifiedGroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

output_dir = Path("/kaggle/working/results/phase2")
output_dir.mkdir(parents=True, exist_ok=True)

data_sources = {
    "nsl_kdd": [
        "/kaggle/input/nsl-kdd/KDDTrain+.txt",
        "/kaggle/input/nsl-kdd/KDDTest+.txt"
    ],
    "cicids2017": [
        "/kaggle/input/cicids2017"
    ],
    "unsw_nb15": [
        "/kaggle/input/unsw-nb15"
    ]
}

split_config = {
    "train_fraction": 0.70,
    "valid_fraction": 0.15,
    "test_fraction": 0.15,
    "random_state": 42
}

attack_config = {
    "gaussian_sigma": 0.05,
    "fgsm_epsilon": 0.05,
    "pgd_epsilon": 0.05,
    "pgd_alpha": 0.01,
    "pgd_steps": 7
}

nsl_columns = [
    "duration", "protocol_type", "service", "flag", "src_bytes", "dst_bytes", "land",
    "wrong_fragment", "urgent", "hot", "num_failed_logins", "logged_in", "num_compromised",
    "root_shell", "su_attempted", "num_root", "num_file_creations", "num_shells",
    "num_access_files", "num_outbound_cmds", "is_host_login", "is_guest_login", "count",
    "srv_count", "serror_rate", "srv_serror_rate", "rerror_rate", "srv_rerror_rate",
    "same_srv_rate", "diff_srv_rate", "srv_diff_host_rate", "dst_host_count",
    "dst_host_srv_count", "dst_host_same_srv_rate", "dst_host_diff_srv_rate",
    "dst_host_same_src_port_rate", "dst_host_srv_diff_host_rate", "dst_host_serror_rate",
    "dst_host_srv_serror_rate", "dst_host_rerror_rate", "dst_host_srv_rerror_rate",
    "raw_label", "difficulty"
]

nsl_family_map = {
    "normal": "normal",
    "back": "dos",
    "land": "dos",
    "neptune": "dos",
    "pod": "dos",
    "smurf": "dos",
    "teardrop": "dos",
    "mailbomb": "dos",
    "apache2": "dos",
    "processtable": "dos",
    "udpstorm": "dos",
    "ipsweep": "probe",
    "nmap": "probe",
    "portsweep": "probe",
    "satan": "probe",
    "mscan": "probe",
    "saint": "probe",
    "ftp_write": "r2l",
    "guess_passwd": "r2l",
    "imap": "r2l",
    "multihop": "r2l",
    "phf": "r2l",
    "spy": "r2l",
    "warezclient": "r2l",
    "warezmaster": "r2l",
    "sendmail": "r2l",
    "named": "r2l",
    "snmpgetattack": "r2l",
    "snmpguess": "r2l",
    "xlock": "r2l",
    "xsnoop": "r2l",
    "worm": "r2l",
    "buffer_overflow": "u2r",
    "loadmodule": "u2r",
    "perl": "u2r",
    "rootkit": "u2r",
    "httptunnel": "u2r",
    "ps": "u2r",
    "sqlattack": "u2r",
    "xterm": "u2r"
}

attack_label_candidates = [
    "label", "labels", "class", "target", "attack", "attack_type", "attack_cat",
    "category", "traffic_label", "raw_label", "rawlabel"
]

family_label_candidates = [
    "attack_family", "family", "attack_cat", "attack_category", "category_group"
]

drop_candidates = {
    "flow_id", "flowid", "src_ip", "dst_ip", "srcip", "dstip", "timestamp", "time", "id",
    "index", "idx", "session_id", "record_id"
}


def normalize_name(value):
    value = str(value).strip().lower()
    value = re.sub(r"[\s\-/]+", "_", value)
    value = re.sub(r"[^a-z0-9_]", "", value)
    value = re.sub(r"_+", "_", value).strip("_")
    return value


def normalize_text(value):
    if pd.isna(value):
        return ""
    value = str(value).strip().lower()
    value = value.replace("\x00", "")
    value = re.sub(r"\s+", " ", value)
    return value


def resolve_files(entries):
    files = []
    for entry in entries:
        path_obj = Path(entry)
        if path_obj.is_dir():
            files.extend(sorted([p for p in path_obj.rglob("*") if p.suffix.lower() in {".csv", ".txt", ".data", ".parquet"}]))
        elif path_obj.exists():
            files.append(path_obj)
    return files


def read_single_file(path_obj):
    suffix = path_obj.suffix.lower()
    if suffix == ".parquet":
        return pd.read_parquet(path_obj)
    try:
        frame = pd.read_csv(path_obj, low_memory=False)
        if frame.shape[1] > 1:
            return frame
    except Exception:
        pass
    try:
        frame = pd.read_csv(path_obj, header=None, low_memory=False)
        if frame.shape[1] > 1:
            return frame
    except Exception:
        pass
    try:
        return pd.read_csv(path_obj, sep=None, engine="python", low_memory=False)
    except Exception:
        return pd.read_csv(path_obj, sep=r"\s+", engine="python", low_memory=False)


def standardize_columns(frame):
    cols = [normalize_name(c) for c in frame.columns]
    used = {}
    final_cols = []
    for col in cols:
        if col not in used:
            used[col] = 0
            final_cols.append(col)
        else:
            used[col] += 1
            final_cols.append(f"{col}_{used[col]}")
    frame.columns = final_cols
    return frame


def assign_known_schema(dataset_key, frame):
    if dataset_key == "nsl_kdd":
        if frame.shape[1] == len(nsl_columns):
            frame.columns = nsl_columns
        elif frame.shape[1] == len(nsl_columns) - 1:
            frame.columns = nsl_columns[:-1]
            frame["difficulty"] = 0
    return frame


def load_dataset(dataset_key, entries):
    parts = []
    for path_obj in resolve_files(entries):
        frame = read_single_file(path_obj)
        frame = assign_known_schema(dataset_key, frame)
        frame = standardize_columns(frame)
        frame["source_file"] = path_obj.name
        parts.append(frame)
    if not parts:
        raise FileNotFoundError(dataset_key)
    frame = pd.concat(parts, axis=0, ignore_index=True)
    frame = standardize_columns(frame)
    return frame


def select_first_present(columns, candidates):
    lookup = {normalize_name(c): c for c in columns}
    for candidate in candidates:
        key = normalize_name(candidate)
        if key in lookup:
            return lookup[key]
    return None


def derive_label_columns(dataset_key, frame):
    label_col = select_first_present(frame.columns, attack_label_candidates)
    family_col = select_first_present(frame.columns, family_label_candidates)

    if dataset_key == "nsl_kdd":
        if "raw_label" in frame.columns:
            label_col = "raw_label"
        family_col = None

    if dataset_key == "unsw_nb15":
        if "attack_cat" in frame.columns:
            family_col = "attack_cat"
        if "label" in frame.columns:
            label_col = "label"

    if dataset_key == "cicids2017":
        if "label" in frame.columns:
            label_col = "label"

    if label_col is None:
        raise ValueError(dataset_key)
    return label_col, family_col


def coerce_basic_types(frame):
    frame = frame.copy()
    frame = frame.replace([np.inf, -np.inf], np.nan)
    for col in frame.columns:
        if frame[col].dtype == object:
            converted = pd.to_numeric(frame[col], errors="coerce")
            if converted.notna().mean() >= 0.90:
                frame[col] = converted
    entirely_missing = [c for c in frame.columns if frame[c].isna().all()]
    if entirely_missing:
        frame = frame.drop(columns=entirely_missing)
    return frame


def drop_redundant_columns(frame, protected):
    columns_to_drop = []
    for col in frame.columns:
        if col in protected:
            continue
        if normalize_name(col) in drop_candidates:
            columns_to_drop.append(col)
            continue
        if frame[col].nunique(dropna=False) <= 1:
            columns_to_drop.append(col)
    if columns_to_drop:
        frame = frame.drop(columns=columns_to_drop)
    return frame


def nsl_family_from_label(raw_value):
    raw_value = normalize_text(raw_value).rstrip(".")
    return nsl_family_map.get(raw_value, "other_attack")


def cicids_family_from_label(raw_value):
    value = normalize_text(raw_value)
    if value in {"benign", "normal"}:
        return "normal"
    if "ddos" in value:
        return "ddos"
    if value.startswith("dos") or "hulk" in value or "goldeneye" in value or "slowhttptest" in value or "slowloris" in value:
        return "dos"
    if "portscan" in value or "port_scan" in value:
        return "portscan"
    if "bot" in value:
        return "botnet"
    if "web attack" in value or "sql injection" in value or "xss" in value or "brute force" in value:
        return "web_attack"
    if "ftp-patator" in value or "ssh-patator" in value:
        return "brute_force"
    if "infiltration" in value:
        return "infiltration"
    if "heartbleed" in value:
        return "heartbleed"
    return normalize_name(value) if value else "unknown"


def unsw_family_from_values(raw_label_value, family_value):
    family_text = normalize_text(family_value)
    if family_text:
        if family_text in {"normal", "benign"}:
            return "normal"
        return normalize_name(family_text)
    raw_text = normalize_text(raw_label_value)
    if raw_text in {"0", "normal", "benign"}:
        return "normal"
    if raw_text in {"1", "attack", "malicious"}:
        return "generic_attack"
    return "unknown"


def derive_family(dataset_key, raw_label_value, family_value=None):
    if dataset_key == "nsl_kdd":
        return nsl_family_from_label(raw_label_value)
    if dataset_key == "cicids2017":
        return cicids_family_from_label(raw_label_value)
    if dataset_key == "unsw_nb15":
        return unsw_family_from_values(raw_label_value, family_value)
    family_text = normalize_text(family_value)
    if family_text in {"normal", "benign"}:
        return "normal"
    if family_text:
        return normalize_name(family_text)
    raw_text = normalize_text(raw_label_value)
    if raw_text in {"0", "normal", "benign"}:
        return "normal"
    if raw_text in {"1", "attack", "malicious"}:
        return "generic_attack"
    return normalize_name(raw_text) if raw_text else "unknown"


def stable_record_hash(frame, feature_columns):
    hash_frame = frame[feature_columns].copy()
    for col in hash_frame.columns:
        if pd.api.types.is_numeric_dtype(hash_frame[col]):
            hash_frame[col] = hash_frame[col].astype("float64").round(8)
        else:
            hash_frame[col] = hash_frame[col].astype(str).str.strip().str.lower()
    return pd.util.hash_pandas_object(hash_frame, index=False).astype(str)


def choose_feature_columns(frame, protected_columns):
    return [c for c in frame.columns if c not in protected_columns]


def make_split_column(frame, stratify_col, group_col, config):
    splitter_outer = StratifiedGroupShuffleSplit(
        n_splits=1,
        test_size=config["test_fraction"],
        random_state=config["random_state"]
    )
    outer_train_idx, test_idx = next(splitter_outer.split(frame, frame[stratify_col], groups=frame[group_col]))

    remainder = frame.iloc[outer_train_idx].copy()
    remainder_y = remainder[stratify_col]
    remainder_groups = remainder[group_col]

    inner_valid_fraction = config["valid_fraction"] / (config["train_fraction"] + config["valid_fraction"])
    splitter_inner = StratifiedGroupShuffleSplit(
        n_splits=1,
        test_size=inner_valid_fraction,
        random_state=config["random_state"]
    )
    train_idx_local, valid_idx_local = next(splitter_inner.split(remainder, remainder_y, groups=remainder_groups))

    frame = frame.copy()
    frame["split_name"] = "train"
    frame.loc[frame.index[test_idx], "split_name"] = "test"
    frame.loc[remainder.index[valid_idx_local], "split_name"] = "valid"
    frame.loc[remainder.index[train_idx_local], "split_name"] = "train"
    return frame


def metric_pack(y_true, y_prob, threshold=0.5):
    y_pred = (y_prob >= threshold).astype(int)
    acc = accuracy_score(y_true, y_pred)
    pre = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    f1v = f1_score(y_true, y_pred, zero_division=0)
    bacc = balanced_accuracy_score(y_true, y_pred)
    try:
        auc = roc_auc_score(y_true, y_prob)
    except Exception:
        auc = np.nan
    try:
        ap = average_precision_score(y_true, y_prob)
    except Exception:
        ap = np.nan
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    fpr = fp / (fp + tn) if (fp + tn) > 0 else np.nan
    return {
        "accuracy": acc,
        "precision": pre,
        "recall": rec,
        "f1": f1v,
        "balanced_accuracy": bacc,
        "roc_auc": auc,
        "pr_auc": ap,
        "false_positive_rate": fpr,
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp)
    }


def build_transformer(frame, feature_columns):
    numeric_columns = [c for c in feature_columns if pd.api.types.is_numeric_dtype(frame[c])]
    categorical_columns = [c for c in feature_columns if c not in numeric_columns]

    numeric_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median"))
    ])

    categorical_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1))
    ])

    transformer = ColumnTransformer(
        transformers=[
            ("num", numeric_pipe, numeric_columns),
            ("cat", categorical_pipe, categorical_columns)
        ],
        remainder="drop"
    )

    return transformer


class DenseModel(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.body = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.20),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.15),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        return self.body(x)


def fit_dense_model(x_train, y_train, x_valid, y_valid, device, epochs=40, batch_size=2048, lr=1e-3, patience=6):
    model = DenseModel(x_train.shape[1]).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    criterion = nn.BCEWithLogitsLoss()

    best_state = copy.deepcopy(model.state_dict())
    best_loss = np.inf
    wait = 0

    indices = np.arange(len(x_train))

    for _ in range(epochs):
        np.random.shuffle(indices)
        model.train()

        for start in range(0, len(indices), batch_size):
            idx = indices[start:start + batch_size]
            xb = torch.from_numpy(x_train[idx]).float().to(device)
            yb = torch.from_numpy(y_train[idx]).float().view(-1, 1).to(device)

            optimizer.zero_grad(set_to_none=True)
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()

        model.eval()
        losses = []

        with torch.no_grad():
            for start in range(0, len(x_valid), batch_size):
                xb = torch.from_numpy(x_valid[start:start + batch_size]).float().to(device)
                yb = torch.from_numpy(y_valid[start:start + batch_size]).float().view(-1, 1).to(device)
                logits = model(xb)
                loss = criterion(logits, yb)
                losses.append(loss.item() * len(xb))

        valid_loss = float(np.sum(losses) / len(x_valid))

        if valid_loss < best_loss - 1e-6:
            best_loss = valid_loss
            best_state = copy.deepcopy(model.state_dict())
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                break

    model.load_state_dict(best_state)
    model.eval()
    return model


def predict_dense_model(model, x_data, device, batch_size=4096):
    outputs = []
    model.eval()
    with torch.no_grad():
        for start in range(0, len(x_data), batch_size):
            xb = torch.from_numpy(x_data[start:start + batch_size]).float().to(device)
            logits = model(xb)
            probs = torch.sigmoid(logits).detach().cpu().numpy().reshape(-1)
            outputs.append(probs)
    return np.concatenate(outputs, axis=0)


def gaussian_corruption(x_data, sigma, clip_min, clip_max, seed=42):
    rng = np.random.default_rng(seed)
    noise = rng.normal(0.0, sigma, size=x_data.shape).astype(np.float32)
    corrupted = x_data + noise
    corrupted = np.clip(corrupted, clip_min, clip_max)
    return corrupted.astype(np.float32)


def fgsm_attack(model, x_data, y_data, epsilon, clip_min, clip_max, device, batch_size=2048):
    criterion = nn.BCEWithLogitsLoss()
    model.eval()
    adv_parts = []
    clip_min_t = torch.from_numpy(clip_min).float().to(device).view(1, -1)
    clip_max_t = torch.from_numpy(clip_max).float().to(device).view(1, -1)

    for start in range(0, len(x_data), batch_size):
        xb = torch.from_numpy(x_data[start:start + batch_size]).float().to(device)
        yb = torch.from_numpy(y_data[start:start + batch_size]).float().view(-1, 1).to(device)
        xb.requires_grad_(True)
        logits = model(xb)
        loss = criterion(logits, yb)
        model.zero_grad(set_to_none=True)
        loss.backward()
        adv = xb + epsilon * xb.grad.sign()
        adv = torch.max(torch.min(adv, clip_max_t), clip_min_t)
        adv_parts.append(adv.detach().cpu().numpy())

    return np.concatenate(adv_parts, axis=0).astype(np.float32)


def pgd_attack(model, x_data, y_data, epsilon, alpha, steps, clip_min, clip_max, device, batch_size=1024):
    criterion = nn.BCEWithLogitsLoss()
    model.eval()
    adv_parts = []
    clip_min_t = torch.from_numpy(clip_min).float().to(device).view(1, -1)
    clip_max_t = torch.from_numpy(clip_max).float().to(device).view(1, -1)

    for start in range(0, len(x_data), batch_size):
        xb = torch.from_numpy(x_data[start:start + batch_size]).float().to(device)
        yb = torch.from_numpy(y_data[start:start + batch_size]).float().view(-1, 1).to(device)

        x0 = xb.detach()
        delta = torch.empty_like(x0).uniform_(-epsilon, epsilon)
        xa = x0 + delta
        xa = torch.max(torch.min(xa, clip_max_t), clip_min_t)

        for _ in range(steps):
            xa.requires_grad_(True)
            logits = model(xa)
            loss = criterion(logits, yb)
            model.zero_grad(set_to_none=True)
            loss.backward()
            grad = xa.grad.detach().sign()
            xa = xa.detach() + alpha * grad
            delta = torch.clamp(xa - x0, min=-epsilon, max=epsilon)
            xa = x0 + delta
            xa = torch.max(torch.min(xa, clip_max_t), clip_min_t)

        adv_parts.append(xa.detach().cpu().numpy())

    return np.concatenate(adv_parts, axis=0).astype(np.float32)


def make_metric_row(scope_name, condition_name, y_true, y_prob, config_dict):
    base = metric_pack(y_true, y_prob, threshold=0.5)
    row = {
        "scope": scope_name,
        "condition": condition_name,
        "sample_count": int(len(y_true)),
        "benign_count": int((np.asarray(y_true) == 0).sum()),
        "attack_count": int((np.asarray(y_true) == 1).sum())
    }
    row.update(base)
    row.update(config_dict)
    return row


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

metric_rows = []
pooled_store = {
    "clean": {"y_true": [], "y_prob": []},
    "gaussian": {"y_true": [], "y_prob": []},
    "fgsm": {"y_true": [], "y_prob": []},
    "pgd": {"y_true": [], "y_prob": []}
}

reference_candidates = []

for dataset_key, entries in data_sources.items():
    raw_frame = load_dataset(dataset_key, entries)
    raw_frame = coerce_basic_types(raw_frame)

    label_col, family_col = derive_label_columns(dataset_key, raw_frame)

    work_frame = raw_frame.copy()
    work_frame["raw_attack_label"] = work_frame[label_col].astype(str).map(normalize_text)

    if family_col is not None and family_col in work_frame.columns:
        family_values = work_frame[family_col]
    else:
        family_values = pd.Series([""] * len(work_frame), index=work_frame.index)

    work_frame["family_label"] = [
        derive_family(dataset_key, raw_val, fam_val)
        for raw_val, fam_val in zip(work_frame["raw_attack_label"], family_values)
    ]
    work_frame["binary_label"] = (work_frame["family_label"] != "normal").astype(int)
    work_frame["dataset_id"] = dataset_key

    protected_before_drop = {"source_file", "raw_attack_label", "family_label", "binary_label", "dataset_id"}
    work_frame = drop_redundant_columns(work_frame, protected_before_drop)

    feature_columns = choose_feature_columns(work_frame, {"source_file", "raw_attack_label", "family_label", "binary_label", "dataset_id"})
    work_frame["record_hash"] = stable_record_hash(work_frame, feature_columns)
    work_frame = work_frame.drop_duplicates(subset=["record_hash"]).reset_index(drop=True)

    feature_columns = choose_feature_columns(work_frame, {"source_file", "raw_attack_label", "family_label", "binary_label", "dataset_id", "record_hash"})
    stratify_source = work_frame["family_label"].copy()
    rare_families = stratify_source.value_counts()
    rare_families = set(rare_families[rare_families < 3].index.tolist())
    stratify_source = stratify_source.apply(lambda x: "rare_attack" if x in rare_families and x != "normal" else x)
    work_frame["stratify_label"] = stratify_source.astype(str)

    work_frame = make_split_column(work_frame, "stratify_label", "record_hash", split_config)

    train_frame = work_frame[work_frame["split_name"] == "train"].copy()
    valid_frame = work_frame[work_frame["split_name"] == "valid"].copy()
    test_frame = work_frame[work_frame["split_name"] == "test"].copy()

    transformer = build_transformer(train_frame, feature_columns)

    x_train_base = transformer.fit_transform(train_frame[feature_columns])
    x_valid_base = transformer.transform(valid_frame[feature_columns])
    x_test_base = transformer.transform(test_frame[feature_columns])

    scaler = StandardScaler()
    x_train = scaler.fit_transform(x_train_base).astype(np.float32)
    x_valid = scaler.transform(x_valid_base).astype(np.float32)
    x_test = scaler.transform(x_test_base).astype(np.float32)

    y_train = train_frame["binary_label"].astype(int).values
    y_valid = valid_frame["binary_label"].astype(int).values
    y_test = test_frame["binary_label"].astype(int).values

    feature_names = transformer.get_feature_names_out().tolist()

    xgb_model = XGBClassifier(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.90,
        colsample_bytree=0.80,
        reg_lambda=1.0,
        min_child_weight=1,
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1
    )
    xgb_model.fit(x_train, y_train)

    valid_prob_xgb = xgb_model.predict_proba(x_valid)[:, 1]
    valid_metrics_xgb = metric_pack(y_valid, valid_prob_xgb)

    reference_candidates.append({
        "dataset_id": dataset_key,
        "selection_score": valid_metrics_xgb["roc_auc"] if not np.isnan(valid_metrics_xgb["roc_auc"]) else valid_metrics_xgb["f1"],
        "model": xgb_model,
        "feature_names": feature_names,
        "x_test": x_test,
        "y_test": y_test
    })

    dense_model = fit_dense_model(x_train, y_train, x_valid, y_valid, device=device)

    clip_min = x_train.min(axis=0).astype(np.float32)
    clip_max = x_train.max(axis=0).astype(np.float32)

    eval_inputs = {}
    eval_inputs["clean"] = x_test
    eval_inputs["gaussian"] = gaussian_corruption(
        x_test,
        sigma=attack_config["gaussian_sigma"],
        clip_min=clip_min,
        clip_max=clip_max,
        seed=42
    )
    eval_inputs["fgsm"] = fgsm_attack(
        dense_model,
        x_test,
        y_test,
        epsilon=attack_config["fgsm_epsilon"],
        clip_min=clip_min,
        clip_max=clip_max,
        device=device
    )
    eval_inputs["pgd"] = pgd_attack(
        dense_model,
        x_test,
        y_test,
        epsilon=attack_config["pgd_epsilon"],
        alpha=attack_config["pgd_alpha"],
        steps=attack_config["pgd_steps"],
        clip_min=clip_min,
        clip_max=clip_max,
        device=device
    )

    for condition_name, x_eval in eval_inputs.items():
        y_prob = predict_dense_model(dense_model, x_eval, device=device)

        config_payload = {
            "gaussian_sigma": attack_config["gaussian_sigma"] if condition_name == "gaussian" else 0.0,
            "fgsm_epsilon": attack_config["fgsm_epsilon"] if condition_name == "fgsm" else 0.0,
            "pgd_epsilon": attack_config["pgd_epsilon"] if condition_name == "pgd" else 0.0,
            "pgd_alpha": attack_config["pgd_alpha"] if condition_name == "pgd" else 0.0,
            "pgd_steps": attack_config["pgd_steps"] if condition_name == "pgd" else 0
        }

        metric_rows.append(make_metric_row(dataset_key, condition_name, y_test, y_prob, config_payload))
        pooled_store[condition_name]["y_true"].append(y_test)
        pooled_store[condition_name]["y_prob"].append(y_prob)

for condition_name, payload in pooled_store.items():
    y_true_all = np.concatenate(payload["y_true"], axis=0)
    y_prob_all = np.concatenate(payload["y_prob"], axis=0)

    config_payload = {
        "gaussian_sigma": attack_config["gaussian_sigma"] if condition_name == "gaussian" else 0.0,
        "fgsm_epsilon": attack_config["fgsm_epsilon"] if condition_name == "fgsm" else 0.0,
        "pgd_epsilon": attack_config["pgd_epsilon"] if condition_name == "pgd" else 0.0,
        "pgd_alpha": attack_config["pgd_alpha"] if condition_name == "pgd" else 0.0,
        "pgd_steps": attack_config["pgd_steps"] if condition_name == "pgd" else 0
    }

    metric_rows.append(make_metric_row("overall", condition_name, y_true_all, y_prob_all, config_payload))

metrics_frame = pd.DataFrame(metric_rows).sort_values(["scope", "condition"]).reset_index(drop=True)
metrics_frame.to_csv(output_dir / "phase2_robustness_table.csv", index=False)

reference_candidates = sorted(reference_candidates, key=lambda x: (-np.nan_to_num(x["selection_score"], nan=-1e9), x["dataset_id"]))
reference_item = reference_candidates[0]

ref_model = reference_item["model"]
ref_feature_names = reference_item["feature_names"]
ref_x_test = reference_item["x_test"]
ref_y_test = reference_item["y_test"]

ref_sample_size = min(2000, len(ref_x_test))
ref_indices = np.random.default_rng(42).choice(len(ref_x_test), size=ref_sample_size, replace=False)
ref_x_sample = ref_x_test[ref_indices]
ref_x_sample_df = pd.DataFrame(ref_x_sample, columns=ref_feature_names)

explainer = shap.TreeExplainer(ref_model)
global_explanation = explainer(ref_x_sample_df)

plt.figure(figsize=(12, 8))
shap.summary_plot(global_explanation.values, ref_x_sample_df, show=False, max_display=20)
plt.tight_layout()
plt.savefig(output_dir / "phase2_shap_global.png", dpi=300, bbox_inches="tight")
plt.close()

ref_prob = ref_model.predict_proba(ref_x_test)[:, 1]
ref_pred = (ref_prob >= 0.5).astype(int)
candidate_idx = np.where((ref_y_test == 1) & (ref_pred == 1))[0]
if len(candidate_idx) == 0:
    candidate_idx = np.where(ref_pred == ref_y_test)[0]
if len(candidate_idx) == 0:
    local_index = int(np.argmin(np.abs(ref_prob - 0.5)))
else:
    candidate_probs = ref_prob[candidate_idx]
    local_index = int(candidate_idx[np.argmin(np.abs(candidate_probs - np.median(candidate_probs)))])

local_df = pd.DataFrame(ref_x_test[local_index:local_index + 1], columns=ref_feature_names)
local_explanation = explainer(local_df)

plt.figure(figsize=(10, 6))
shap.plots.waterfall(local_explanation[0], max_display=15, show=False)
plt.tight_layout()
plt.savefig(output_dir / "phase2_shap_local.png", dpi=300, bbox_inches="tight")
plt.close()

overall_frame = metrics_frame[metrics_frame["scope"] == "overall"].copy()
overall_frame["condition"] = pd.Categorical(overall_frame["condition"], categories=["clean", "gaussian", "fgsm", "pgd"], ordered=True)
overall_frame = overall_frame.sort_values("condition").reset_index(drop=True)

accuracy_frame = overall_frame[overall_frame["condition"].isin(["clean", "fgsm", "pgd"])].copy()

plt.figure(figsize=(8, 5))
plt.bar(accuracy_frame["condition"].tolist(), accuracy_frame["accuracy"].tolist())
plt.ylim(0.0, 1.0)
plt.ylabel("Accuracy")
plt.xlabel("Condition")
plt.tight_layout()
plt.savefig(output_dir / "phase2_accuracy_under_attacks.png", dpi=300, bbox_inches="tight")
plt.close()

gaussian_true = np.concatenate(pooled_store["gaussian"]["y_true"], axis=0)
gaussian_prob = np.concatenate(pooled_store["gaussian"]["y_prob"], axis=0)
gaussian_pred = (gaussian_prob >= 0.5).astype(int)

p_vals, r_vals, f_vals, _ = precision_recall_fscore_support(
    gaussian_true,
    gaussian_pred,
    labels=[0, 1],
    zero_division=0
)

heatmap_values = np.array([
    [p_vals[0], r_vals[0], f_vals[0]],
    [p_vals[1], r_vals[1], f_vals[1]]
])

plt.figure(figsize=(6, 4))
plt.imshow(heatmap_values, aspect="auto")
plt.xticks([0, 1, 2], ["Precision", "Recall", "F1"])
plt.yticks([0, 1], ["Benign", "Attack"])
for i in range(heatmap_values.shape[0]):
    for j in range(heatmap_values.shape[1]):
        plt.text(j, i, f"{heatmap_values[i, j]:.3f}", ha="center", va="center")
plt.colorbar()
plt.tight_layout()
plt.savefig(output_dir / "phase2_gaussian_heatmap.png", dpi=300, bbox_inches="tight")
plt.close()

comparison_metrics = ["accuracy", "precision", "recall", "f1"]
plot_frame = overall_frame[["condition"] + comparison_metrics].copy()
x_positions = np.arange(len(plot_frame))
bar_width = 0.18

plt.figure(figsize=(10, 6))
for idx, metric_name in enumerate(comparison_metrics):
    plt.bar(x_positions + idx * bar_width, plot_frame[metric_name].values, width=bar_width, label=metric_name)
plt.xticks(x_positions + 1.5 * bar_width, plot_frame["condition"].tolist())
plt.ylim(0.0, 1.0)
plt.xlabel("Condition")
plt.ylabel("Score")
plt.legend()
plt.tight_layout()
plt.savefig(output_dir / "phase2_robustness_comparison.png", dpi=300, bbox_inches="tight")
plt.close()

In [ ]:
from pathlib import Path
import copy
import json
import random
import re
import warnings

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import StratifiedGroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder, StandardScaler

warnings.filterwarnings("ignore")

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

output_dir = Path("/kaggle/working/results/phase3")
output_dir.mkdir(parents=True, exist_ok=True)

data_sources = {
    "nsl_kdd": [
        "/kaggle/input/nsl-kdd/KDDTrain+.txt",
        "/kaggle/input/nsl-kdd/KDDTest+.txt"
    ],
    "cicids2017": [
        "/kaggle/input/cicids2017"
    ],
    "unsw_nb15": [
        "/kaggle/input/unsw-nb15"
    ]
}

split_config = {
    "train_fraction": 0.70,
    "valid_fraction": 0.15,
    "test_fraction": 0.15,
    "random_state": 42
}

train_config = {
    "epochs": 40,
    "batch_size": 2048,
    "learning_rate": 1e-3,
    "weight_decay": 1e-4,
    "patience": 8
}

nsl_columns = [
    "duration", "protocol_type", "service", "flag", "src_bytes", "dst_bytes", "land",
    "wrong_fragment", "urgent", "hot", "num_failed_logins", "logged_in", "num_compromised",
    "root_shell", "su_attempted", "num_root", "num_file_creations", "num_shells",
    "num_access_files", "num_outbound_cmds", "is_host_login", "is_guest_login", "count",
    "srv_count", "serror_rate", "srv_serror_rate", "rerror_rate", "srv_rerror_rate",
    "same_srv_rate", "diff_srv_rate", "srv_diff_host_rate", "dst_host_count",
    "dst_host_srv_count", "dst_host_same_srv_rate", "dst_host_diff_srv_rate",
    "dst_host_same_src_port_rate", "dst_host_srv_diff_host_rate", "dst_host_serror_rate",
    "dst_host_srv_serror_rate", "dst_host_rerror_rate", "dst_host_srv_rerror_rate",
    "raw_label", "difficulty"
]

nsl_family_map = {
    "normal": "normal",
    "back": "dos",
    "land": "dos",
    "neptune": "dos",
    "pod": "dos",
    "smurf": "dos",
    "teardrop": "dos",
    "mailbomb": "dos",
    "apache2": "dos",
    "processtable": "dos",
    "udpstorm": "dos",
    "ipsweep": "probe",
    "nmap": "probe",
    "portsweep": "probe",
    "satan": "probe",
    "mscan": "probe",
    "saint": "probe",
    "ftp_write": "r2l",
    "guess_passwd": "r2l",
    "imap": "r2l",
    "multihop": "r2l",
    "phf": "r2l",
    "spy": "r2l",
    "warezclient": "r2l",
    "warezmaster": "r2l",
    "sendmail": "r2l",
    "named": "r2l",
    "snmpgetattack": "r2l",
    "snmpguess": "r2l",
    "xlock": "r2l",
    "xsnoop": "r2l",
    "worm": "r2l",
    "buffer_overflow": "u2r",
    "loadmodule": "u2r",
    "perl": "u2r",
    "rootkit": "u2r",
    "httptunnel": "u2r",
    "ps": "u2r",
    "sqlattack": "u2r",
    "xterm": "u2r"
}

attack_label_candidates = [
    "label", "labels", "class", "target", "attack", "attack_type", "attack_cat",
    "category", "traffic_label", "raw_label", "rawlabel"
]

family_label_candidates = [
    "attack_family", "family", "attack_cat", "attack_category", "category_group"
]

drop_candidates = {
    "flow_id", "flowid", "src_ip", "dst_ip", "srcip", "dstip", "timestamp", "time", "id",
    "index", "idx", "session_id", "record_id"
}


def normalize_name(value):
    value = str(value).strip().lower()
    value = re.sub(r"[\s\-/]+", "_", value)
    value = re.sub(r"[^a-z0-9_]", "", value)
    value = re.sub(r"_+", "_", value).strip("_")
    return value


def normalize_text(value):
    if pd.isna(value):
        return ""
    value = str(value).strip().lower()
    value = value.replace("\x00", "")
    value = re.sub(r"\s+", " ", value)
    return value


def resolve_files(entries):
    files = []
    for entry in entries:
        path_obj = Path(entry)
        if path_obj.is_dir():
            files.extend(sorted([p for p in path_obj.rglob("*") if p.suffix.lower() in {".csv", ".txt", ".data", ".parquet"}]))
        elif path_obj.exists():
            files.append(path_obj)
    return files


def read_single_file(path_obj):
    suffix = path_obj.suffix.lower()
    if suffix == ".parquet":
        return pd.read_parquet(path_obj)
    try:
        frame = pd.read_csv(path_obj, low_memory=False)
        if frame.shape[1] > 1:
            return frame
    except Exception:
        pass
    try:
        frame = pd.read_csv(path_obj, header=None, low_memory=False)
        if frame.shape[1] > 1:
            return frame
    except Exception:
        pass
    try:
        return pd.read_csv(path_obj, sep=None, engine="python", low_memory=False)
    except Exception:
        return pd.read_csv(path_obj, sep=r"\s+", engine="python", low_memory=False)


def standardize_columns(frame):
    cols = [normalize_name(c) for c in frame.columns]
    used = {}
    final_cols = []
    for col in cols:
        if col not in used:
            used[col] = 0
            final_cols.append(col)
        else:
            used[col] += 1
            final_cols.append(f"{col}_{used[col]}")
    frame.columns = final_cols
    return frame


def assign_known_schema(dataset_key, frame):
    if dataset_key == "nsl_kdd":
        if frame.shape[1] == len(nsl_columns):
            frame.columns = nsl_columns
        elif frame.shape[1] == len(nsl_columns) - 1:
            frame.columns = nsl_columns[:-1]
            frame["difficulty"] = 0
    return frame


def load_dataset(dataset_key, entries):
    parts = []
    for path_obj in resolve_files(entries):
        frame = read_single_file(path_obj)
        frame = assign_known_schema(dataset_key, frame)
        frame = standardize_columns(frame)
        frame["source_file"] = path_obj.name
        parts.append(frame)
    if not parts:
        raise FileNotFoundError(dataset_key)
    frame = pd.concat(parts, axis=0, ignore_index=True)
    frame = standardize_columns(frame)
    return frame


def select_first_present(columns, candidates):
    lookup = {normalize_name(c): c for c in columns}
    for candidate in candidates:
        key = normalize_name(candidate)
        if key in lookup:
            return lookup[key]
    return None


def derive_label_columns(dataset_key, frame):
    label_col = select_first_present(frame.columns, attack_label_candidates)
    family_col = select_first_present(frame.columns, family_label_candidates)

    if dataset_key == "nsl_kdd":
        if "raw_label" in frame.columns:
            label_col = "raw_label"
        family_col = None

    if dataset_key == "unsw_nb15":
        if "attack_cat" in frame.columns:
            family_col = "attack_cat"
        if "label" in frame.columns:
            label_col = "label"

    if dataset_key == "cicids2017":
        if "label" in frame.columns:
            label_col = "label"

    if label_col is None:
        raise ValueError(dataset_key)
    return label_col, family_col


def coerce_basic_types(frame):
    frame = frame.copy()
    frame = frame.replace([np.inf, -np.inf], np.nan)
    for col in frame.columns:
        if frame[col].dtype == object:
            converted = pd.to_numeric(frame[col], errors="coerce")
            if converted.notna().mean() >= 0.90:
                frame[col] = converted
    entirely_missing = [c for c in frame.columns if frame[c].isna().all()]
    if entirely_missing:
        frame = frame.drop(columns=entirely_missing)
    return frame


def drop_redundant_columns(frame, protected):
    columns_to_drop = []
    for col in frame.columns:
        if col in protected:
            continue
        if normalize_name(col) in drop_candidates:
            columns_to_drop.append(col)
            continue
        if frame[col].nunique(dropna=False) <= 1:
            columns_to_drop.append(col)
    if columns_to_drop:
        frame = frame.drop(columns=columns_to_drop)
    return frame


def nsl_family_from_label(raw_value):
    raw_value = normalize_text(raw_value).rstrip(".")
    return nsl_family_map.get(raw_value, "other_attack")


def cicids_family_from_label(raw_value):
    value = normalize_text(raw_value)
    if value in {"benign", "normal"}:
        return "normal"
    if "ddos" in value:
        return "ddos"
    if value.startswith("dos") or "hulk" in value or "goldeneye" in value or "slowhttptest" in value or "slowloris" in value:
        return "dos"
    if "portscan" in value or "port_scan" in value:
        return "portscan"
    if "bot" in value:
        return "botnet"
    if "web attack" in value or "sql injection" in value or "xss" in value or "brute force" in value:
        return "web_attack"
    if "ftp-patator" in value or "ssh-patator" in value:
        return "brute_force"
    if "infiltration" in value:
        return "infiltration"
    if "heartbleed" in value:
        return "heartbleed"
    return normalize_name(value) if value else "unknown"


def unsw_family_from_values(raw_label_value, family_value):
    family_text = normalize_text(family_value)
    if family_text:
        if family_text in {"normal", "benign"}:
            return "normal"
        return normalize_name(family_text)
    raw_text = normalize_text(raw_label_value)
    if raw_text in {"0", "normal", "benign"}:
        return "normal"
    if raw_text in {"1", "attack", "malicious"}:
        return "generic_attack"
    return "unknown"


def derive_family(dataset_key, raw_label_value, family_value=None):
    if dataset_key == "nsl_kdd":
        return nsl_family_from_label(raw_label_value)
    if dataset_key == "cicids2017":
        return cicids_family_from_label(raw_label_value)
    if dataset_key == "unsw_nb15":
        return unsw_family_from_values(raw_label_value, family_value)
    family_text = normalize_text(family_value)
    if family_text in {"normal", "benign"}:
        return "normal"
    if family_text:
        return normalize_name(family_text)
    raw_text = normalize_text(raw_label_value)
    if raw_text in {"0", "normal", "benign"}:
        return "normal"
    if raw_text in {"1", "attack", "malicious"}:
        return "generic_attack"
    return normalize_name(raw_text) if raw_text else "unknown"


def stable_record_hash(frame, feature_columns):
    hash_frame = frame[feature_columns].copy()
    for col in hash_frame.columns:
        if pd.api.types.is_numeric_dtype(hash_frame[col]):
            hash_frame[col] = hash_frame[col].astype("float64").round(8)
        else:
            hash_frame[col] = hash_frame[col].astype(str).str.strip().str.lower()
    return pd.util.hash_pandas_object(hash_frame, index=False).astype(str)


def choose_feature_columns(frame, protected_columns):
    return [c for c in frame.columns if c not in protected_columns]


def make_split_column(frame, stratify_col, group_col, config):
    splitter_outer = StratifiedGroupShuffleSplit(
        n_splits=1,
        test_size=config["test_fraction"],
        random_state=config["random_state"]
    )
    outer_train_idx, test_idx = next(splitter_outer.split(frame, frame[stratify_col], groups=frame[group_col]))

    remainder = frame.iloc[outer_train_idx].copy()
    remainder_y = remainder[stratify_col]
    remainder_groups = remainder[group_col]

    inner_valid_fraction = config["valid_fraction"] / (config["train_fraction"] + config["valid_fraction"])
    splitter_inner = StratifiedGroupShuffleSplit(
        n_splits=1,
        test_size=inner_valid_fraction,
        random_state=config["random_state"]
    )
    train_idx_local, valid_idx_local = next(splitter_inner.split(remainder, remainder_y, groups=remainder_groups))

    frame = frame.copy()
    frame["split_name"] = "train"
    frame.loc[frame.index[test_idx], "split_name"] = "test"
    frame.loc[remainder.index[valid_idx_local], "split_name"] = "valid"
    frame.loc[remainder.index[train_idx_local], "split_name"] = "train"
    return frame


def build_transformer(frame, feature_columns):
    numeric_columns = [c for c in feature_columns if pd.api.types.is_numeric_dtype(frame[c])]
    categorical_columns = [c for c in feature_columns if c not in numeric_columns]

    numeric_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median"))
    ])

    categorical_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1))
    ])

    transformer = ColumnTransformer(
        transformers=[
            ("num", numeric_pipe, numeric_columns),
            ("cat", categorical_pipe, categorical_columns)
        ],
        remainder="drop"
    )

    return transformer


class DenseModel(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.body = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.20),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.15),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        return self.body(x)


def evaluate_epoch(model, x_data, y_data, device, batch_size=4096):
    criterion = nn.BCEWithLogitsLoss()
    losses = []
    probs_all = []

    model.eval()
    with torch.no_grad():
        for start in range(0, len(x_data), batch_size):
            xb = torch.from_numpy(x_data[start:start + batch_size]).float().to(device)
            yb = torch.from_numpy(y_data[start:start + batch_size]).float().view(-1, 1).to(device)
            logits = model(xb)
            loss = criterion(logits, yb)
            probs = torch.sigmoid(logits).detach().cpu().numpy().reshape(-1)
            losses.append(loss.item() * len(xb))
            probs_all.append(probs)

    probs_all = np.concatenate(probs_all, axis=0)
    preds = (probs_all >= 0.5).astype(int)
    avg_loss = float(np.sum(losses) / len(x_data))
    acc = accuracy_score(y_data, preds)
    prec = precision_score(y_data, preds, zero_division=0)
    rec = recall_score(y_data, preds, zero_division=0)
    f1v = f1_score(y_data, preds, zero_division=0)

    try:
        auc = roc_auc_score(y_data, probs_all)
    except Exception:
        auc = np.nan

    return {
        "loss": avg_loss,
        "accuracy": acc,
        "precision": prec,
        "recall": rec,
        "f1": f1v,
        "roc_auc": auc
    }


def fit_with_log(x_train, y_train, x_valid, y_valid, device, epochs, batch_size, lr, weight_decay, patience):
    model = DenseModel(x_train.shape[1]).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    criterion = nn.BCEWithLogitsLoss()

    best_state = copy.deepcopy(model.state_dict())
    best_loss = np.inf
    best_epoch = 0
    wait = 0
    history = []

    indices = np.arange(len(x_train))

    for epoch in range(1, epochs + 1):
        np.random.shuffle(indices)
        model.train()
        running_loss = 0.0

        for start in range(0, len(indices), batch_size):
            idx = indices[start:start + batch_size]
            xb = torch.from_numpy(x_train[idx]).float().to(device)
            yb = torch.from_numpy(y_train[idx]).float().view(-1, 1).to(device)

            optimizer.zero_grad(set_to_none=True)
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * len(xb)

        train_loss_only = float(running_loss / len(x_train))
        train_metrics = evaluate_epoch(model, x_train, y_train, device=device, batch_size=batch_size)
        valid_metrics = evaluate_epoch(model, x_valid, y_valid, device=device, batch_size=batch_size)

        row = {
            "epoch": epoch,
            "train_loss_batch": train_loss_only,
            "train_loss": train_metrics["loss"],
            "valid_loss": valid_metrics["loss"],
            "train_accuracy": train_metrics["accuracy"],
            "valid_accuracy": valid_metrics["accuracy"],
            "train_precision": train_metrics["precision"],
            "valid_precision": valid_metrics["precision"],
            "train_recall": train_metrics["recall"],
            "valid_recall": valid_metrics["recall"],
            "train_f1": train_metrics["f1"],
            "valid_f1": valid_metrics["f1"],
            "train_roc_auc": train_metrics["roc_auc"],
            "valid_roc_auc": valid_metrics["roc_auc"]
        }
        history.append(row)

        if valid_metrics["loss"] < best_loss - 1e-6:
            best_loss = valid_metrics["loss"]
            best_state = copy.deepcopy(model.state_dict())
            best_epoch = epoch
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                break

    model.load_state_dict(best_state)
    return model, pd.DataFrame(history), best_epoch


all_frames = []

for dataset_key, entries in data_sources.items():
    raw_frame = load_dataset(dataset_key, entries)
    raw_frame = coerce_basic_types(raw_frame)
    label_col, family_col = derive_label_columns(dataset_key, raw_frame)

    work_frame = raw_frame.copy()
    work_frame["raw_attack_label"] = work_frame[label_col].astype(str).map(normalize_text)

    if family_col is not None and family_col in work_frame.columns:
        family_values = work_frame[family_col]
    else:
        family_values = pd.Series([""] * len(work_frame), index=work_frame.index)

    work_frame["family_label"] = [
        derive_family(dataset_key, raw_val, fam_val)
        for raw_val, fam_val in zip(work_frame["raw_attack_label"], family_values)
    ]
    work_frame["binary_label"] = (work_frame["family_label"] != "normal").astype(int)
    work_frame["dataset_id"] = dataset_key

    protected_before_drop = {"source_file", "raw_attack_label", "family_label", "binary_label", "dataset_id"}
    work_frame = drop_redundant_columns(work_frame, protected_before_drop)

    feature_columns = choose_feature_columns(work_frame, {"source_file", "raw_attack_label", "family_label", "binary_label", "dataset_id"})
    work_frame["record_hash"] = stable_record_hash(work_frame, feature_columns)
    work_frame = work_frame.drop_duplicates(subset=["record_hash"]).reset_index(drop=True)

    feature_columns = choose_feature_columns(work_frame, {"source_file", "raw_attack_label", "family_label", "binary_label", "dataset_id", "record_hash"})
    stratify_source = work_frame["family_label"].copy()
    rare_families = stratify_source.value_counts()
    rare_families = set(rare_families[rare_families < 3].index.tolist())
    stratify_source = stratify_source.apply(lambda x: "rare_attack" if x in rare_families and x != "normal" else x)
    work_frame["stratify_label"] = stratify_source.astype(str)

    work_frame = make_split_column(work_frame, "stratify_label", "record_hash", split_config)
    all_frames.append(work_frame)

full_frame = pd.concat(all_frames, axis=0, ignore_index=True)

feature_columns = choose_feature_columns(
    full_frame,
    {
        "source_file", "raw_attack_label", "family_label", "binary_label", "dataset_id",
        "record_hash", "stratify_label", "split_name"
    }
)

train_frame = full_frame[full_frame["split_name"] == "train"].copy()
valid_frame = full_frame[full_frame["split_name"] == "valid"].copy()
test_frame = full_frame[full_frame["split_name"] == "test"].copy()

transformer = build_transformer(train_frame, feature_columns)

x_train_base = transformer.fit_transform(train_frame[feature_columns])
x_valid_base = transformer.transform(valid_frame[feature_columns])
x_test_base = transformer.transform(test_frame[feature_columns])

scaler = StandardScaler()
x_train = scaler.fit_transform(x_train_base).astype(np.float32)
x_valid = scaler.transform(x_valid_base).astype(np.float32)
x_test = scaler.transform(x_test_base).astype(np.float32)

y_train = train_frame["binary_label"].astype(int).values
y_valid = valid_frame["binary_label"].astype(int).values
y_test = test_frame["binary_label"].astype(int).values

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model, history_frame, best_epoch = fit_with_log(
    x_train=x_train,
    y_train=y_train,
    x_valid=x_valid,
    y_valid=y_valid,
    device=device,
    epochs=train_config["epochs"],
    batch_size=train_config["batch_size"],
    lr=train_config["learning_rate"],
    weight_decay=train_config["weight_decay"],
    patience=train_config["patience"]
)

test_metrics = evaluate_epoch(model, x_test, y_test, device=device, batch_size=train_config["batch_size"])
best_row = history_frame.loc[history_frame["epoch"] == best_epoch].iloc[0]

history_frame.to_csv(output_dir / "phase3_training_log.csv", index=False)

plt.figure(figsize=(8, 5))
plt.plot(history_frame["epoch"], history_frame["train_accuracy"], label="Train")
plt.plot(history_frame["epoch"], history_frame["valid_accuracy"], label="Validation")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.tight_layout()
plt.savefig(output_dir / "phase3_learning_curve_accuracy.png", dpi=300, bbox_inches="tight")
plt.close()

plt.figure(figsize=(8, 5))
plt.plot(history_frame["epoch"], history_frame["train_loss"], label="Train")
plt.plot(history_frame["epoch"], history_frame["valid_loss"], label="Validation")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.tight_layout()
plt.savefig(output_dir / "phase3_learning_curve_loss.png", dpi=300, bbox_inches="tight")
plt.close()

summary_lines = []
summary_lines.append(
    f"The convergence analysis was conducted under the same leakage-controlled preprocessing and split policy used in the earlier phases, with pooled training across the harmonized benchmarks and validation-based early stopping."
)
summary_lines.append(
    f"Training ran for {int(history_frame['epoch'].max())} epochs, and the best validation loss was reached at epoch {int(best_epoch)} with validation accuracy {best_row['valid_accuracy']:.4f}, validation F1 {best_row['valid_f1']:.4f}, and validation ROC-AUC {best_row['valid_roc_auc']:.4f}."
)
summary_lines.append(
    f"At the same selected checkpoint, the training accuracy was {best_row['train_accuracy']:.4f} and the training loss was {best_row['train_loss']:.4f}, while the validation loss was {best_row['valid_loss']:.4f}, indicating the final operating point used for model selection."
)
summary_lines.append(
    f"The held-out test split evaluated at this checkpoint achieved accuracy {test_metrics['accuracy']:.4f}, precision {test_metrics['precision']:.4f}, recall {test_metrics['recall']:.4f}, F1 {test_metrics['f1']:.4f}, and ROC-AUC {test_metrics['roc_auc']:.4f}."
)
summary_lines.append(
    f"Overall, the learning curves provide compact evidence that optimization stabilized without relying on the final epoch alone, and the saved epoch log supports direct manuscript reporting of convergence behavior."
)

with open(output_dir / "phase3_convergence_summary.txt", "w", encoding="utf-8") as handle:
    handle.write("\n".join(summary_lines))

In [ ]:
from pathlib import Path
import copy
import random
import re
import warnings

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, average_precision_score, balanced_accuracy_score, confusion_matrix, precision_recall_curve, precision_recall_fscore_support, precision_score, recall_score, roc_auc_score, roc_curve, f1_score
from sklearn.model_selection import StratifiedGroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder, StandardScaler

warnings.filterwarnings("ignore")

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

output_dir = Path("/kaggle/working/results/phase4")
output_dir.mkdir(parents=True, exist_ok=True)

data_sources = {
    "nsl_kdd": {
        "train": [
            "/kaggle/input/nsl-kdd/KDDTrain+.txt"
        ],
        "test": [
            "/kaggle/input/nsl-kdd/KDDTest+.txt"
        ]
    },
    "cicids2017": [
        "/kaggle/input/cicids2017"
    ],
    "unsw_nb15": [
        "/kaggle/input/unsw-nb15"
    ]
}

split_config = {
    "train_fraction": 0.85,
    "valid_fraction": 0.15,
    "random_state": 42
}

gaussian_config = {
    "sigma": 0.05
}

train_config = {
    "epochs": 40,
    "batch_size": 2048,
    "learning_rate": 1e-3,
    "weight_decay": 1e-4,
    "patience": 8
}

nsl_columns = [
    "duration", "protocol_type", "service", "flag", "src_bytes", "dst_bytes", "land",
    "wrong_fragment", "urgent", "hot", "num_failed_logins", "logged_in", "num_compromised",
    "root_shell", "su_attempted", "num_root", "num_file_creations", "num_shells",
    "num_access_files", "num_outbound_cmds", "is_host_login", "is_guest_login", "count",
    "srv_count", "serror_rate", "srv_serror_rate", "rerror_rate", "srv_rerror_rate",
    "same_srv_rate", "diff_srv_rate", "srv_diff_host_rate", "dst_host_count",
    "dst_host_srv_count", "dst_host_same_srv_rate", "dst_host_diff_srv_rate",
    "dst_host_same_src_port_rate", "dst_host_srv_diff_host_rate", "dst_host_serror_rate",
    "dst_host_srv_serror_rate", "dst_host_rerror_rate", "dst_host_srv_rerror_rate",
    "raw_label", "difficulty"
]

nsl_family_map = {
    "normal": "normal",
    "back": "dos",
    "land": "dos",
    "neptune": "dos",
    "pod": "dos",
    "smurf": "dos",
    "teardrop": "dos",
    "mailbomb": "dos",
    "apache2": "dos",
    "processtable": "dos",
    "udpstorm": "dos",
    "ipsweep": "probe",
    "nmap": "probe",
    "portsweep": "probe",
    "satan": "probe",
    "mscan": "probe",
    "saint": "probe",
    "ftp_write": "r2l",
    "guess_passwd": "r2l",
    "imap": "r2l",
    "multihop": "r2l",
    "phf": "r2l",
    "spy": "r2l",
    "warezclient": "r2l",
    "warezmaster": "r2l",
    "sendmail": "r2l",
    "named": "r2l",
    "snmpgetattack": "r2l",
    "snmpguess": "r2l",
    "xlock": "r2l",
    "xsnoop": "r2l",
    "worm": "r2l",
    "buffer_overflow": "u2r",
    "loadmodule": "u2r",
    "perl": "u2r",
    "rootkit": "u2r",
    "httptunnel": "u2r",
    "ps": "u2r",
    "sqlattack": "u2r",
    "xterm": "u2r"
}

attack_label_candidates = [
    "label", "labels", "class", "target", "attack", "attack_type", "attack_cat",
    "category", "traffic_label", "raw_label", "rawlabel"
]

family_label_candidates = [
    "attack_family", "family", "attack_cat", "attack_category", "category_group"
]

drop_candidates = {
    "flow_id", "flowid", "src_ip", "dst_ip", "srcip", "dstip", "timestamp", "time", "id",
    "index", "idx", "session_id", "record_id"
}


def normalize_name(value):
    value = str(value).strip().lower()
    value = re.sub(r"[\s\-/]+", "_", value)
    value = re.sub(r"[^a-z0-9_]", "", value)
    value = re.sub(r"_+", "_", value).strip("_")
    return value


def normalize_text(value):
    if pd.isna(value):
        return ""
    value = str(value).strip().lower()
    value = value.replace("\x00", "")
    value = re.sub(r"\s+", " ", value)
    return value


def resolve_files(entries):
    files = []
    for entry in entries:
        path_obj = Path(entry)
        if path_obj.is_dir():
            files.extend(sorted([p for p in path_obj.rglob("*") if p.suffix.lower() in {".csv", ".txt", ".data", ".parquet"}]))
        elif path_obj.exists():
            files.append(path_obj)
    return files


def read_single_file(path_obj):
    suffix = path_obj.suffix.lower()
    if suffix == ".parquet":
        return pd.read_parquet(path_obj)
    try:
        frame = pd.read_csv(path_obj, low_memory=False)
        if frame.shape[1] > 1:
            return frame
    except Exception:
        pass
    try:
        frame = pd.read_csv(path_obj, header=None, low_memory=False)
        if frame.shape[1] > 1:
            return frame
    except Exception:
        pass
    try:
        return pd.read_csv(path_obj, sep=None, engine="python", low_memory=False)
    except Exception:
        return pd.read_csv(path_obj, sep=r"\s+", engine="python", low_memory=False)


def standardize_columns(frame):
    cols = [normalize_name(c) for c in frame.columns]
    used = {}
    final_cols = []
    for col in cols:
        if col not in used:
            used[col] = 0
            final_cols.append(col)
        else:
            used[col] += 1
            final_cols.append(f"{col}_{used[col]}")
    frame.columns = final_cols
    return frame


def assign_known_schema(dataset_key, frame):
    if dataset_key == "nsl_kdd":
        if frame.shape[1] == len(nsl_columns):
            frame.columns = nsl_columns
        elif frame.shape[1] == len(nsl_columns) - 1:
            frame.columns = nsl_columns[:-1]
            frame["difficulty"] = 0
    return frame


def load_dataset(dataset_key, entries):
    parts = []
    for path_obj in resolve_files(entries):
        frame = read_single_file(path_obj)
        frame = assign_known_schema(dataset_key, frame)
        frame = standardize_columns(frame)
        frame["source_file"] = path_obj.name
        parts.append(frame)
    if not parts:
        raise FileNotFoundError(dataset_key)
    frame = pd.concat(parts, axis=0, ignore_index=True)
    frame = standardize_columns(frame)
    return frame


def select_first_present(columns, candidates):
    lookup = {normalize_name(c): c for c in columns}
    for candidate in candidates:
        key = normalize_name(candidate)
        if key in lookup:
            return lookup[key]
    return None


def derive_label_columns(dataset_key, frame):
    label_col = select_first_present(frame.columns, attack_label_candidates)
    family_col = select_first_present(frame.columns, family_label_candidates)

    if dataset_key == "nsl_kdd":
        if "raw_label" in frame.columns:
            label_col = "raw_label"
        family_col = None

    if dataset_key == "unsw_nb15":
        if "attack_cat" in frame.columns:
            family_col = "attack_cat"
        if "label" in frame.columns:
            label_col = "label"

    if dataset_key == "cicids2017":
        if "label" in frame.columns:
            label_col = "label"

    if label_col is None:
        raise ValueError(dataset_key)
    return label_col, family_col


def coerce_basic_types(frame):
    frame = frame.copy()
    frame = frame.replace([np.inf, -np.inf], np.nan)
    for col in frame.columns:
        if frame[col].dtype == object:
            converted = pd.to_numeric(frame[col], errors="coerce")
            if converted.notna().mean() >= 0.90:
                frame[col] = converted
    entirely_missing = [c for c in frame.columns if frame[c].isna().all()]
    if entirely_missing:
        frame = frame.drop(columns=entirely_missing)
    return frame


def drop_redundant_columns(frame, protected):
    columns_to_drop = []
    for col in frame.columns:
        if col in protected:
            continue
        if normalize_name(col) in drop_candidates:
            columns_to_drop.append(col)
            continue
        if frame[col].nunique(dropna=False) <= 1:
            columns_to_drop.append(col)
    if columns_to_drop:
        frame = frame.drop(columns=columns_to_drop)
    return frame


def nsl_family_from_label(raw_value):
    raw_value = normalize_text(raw_value).rstrip(".")
    return nsl_family_map.get(raw_value, "other_attack")


def cicids_family_from_label(raw_value):
    value = normalize_text(raw_value)
    if value in {"benign", "normal"}:
        return "normal"
    if "ddos" in value:
        return "ddos"
    if value.startswith("dos") or "hulk" in value or "goldeneye" in value or "slowhttptest" in value or "slowloris" in value:
        return "dos"
    if "portscan" in value or "port_scan" in value:
        return "portscan"
    if "bot" in value:
        return "botnet"
    if "web attack" in value or "sql injection" in value or "xss" in value or "brute force" in value:
        return "web_attack"
    if "ftp-patator" in value or "ssh-patator" in value:
        return "brute_force"
    if "infiltration" in value:
        return "infiltration"
    if "heartbleed" in value:
        return "heartbleed"
    return normalize_name(value) if value else "unknown"


def unsw_family_from_values(raw_label_value, family_value):
    family_text = normalize_text(family_value)
    if family_text:
        if family_text in {"normal", "benign"}:
            return "normal"
        return normalize_name(family_text)
    raw_text = normalize_text(raw_label_value)
    if raw_text in {"0", "normal", "benign"}:
        return "normal"
    if raw_text in {"1", "attack", "malicious"}:
        return "generic_attack"
    return "unknown"


def derive_family(dataset_key, raw_label_value, family_value=None):
    if dataset_key == "nsl_kdd":
        return nsl_family_from_label(raw_label_value)
    if dataset_key == "cicids2017":
        return cicids_family_from_label(raw_label_value)
    if dataset_key == "unsw_nb15":
        return unsw_family_from_values(raw_label_value, family_value)
    family_text = normalize_text(family_value)
    if family_text in {"normal", "benign"}:
        return "normal"
    if family_text:
        return normalize_name(family_text)
    raw_text = normalize_text(raw_label_value)
    if raw_text in {"0", "normal", "benign"}:
        return "normal"
    if raw_text in {"1", "attack", "malicious"}:
        return "generic_attack"
    return normalize_name(raw_text) if raw_text else "unknown"


def stable_record_hash(frame, feature_columns):
    hash_frame = frame[feature_columns].copy()
    for col in hash_frame.columns:
        if pd.api.types.is_numeric_dtype(hash_frame[col]):
            hash_frame[col] = hash_frame[col].astype("float64").round(8)
        else:
            hash_frame[col] = hash_frame[col].astype(str).str.strip().str.lower()
    return pd.util.hash_pandas_object(hash_frame, index=False).astype(str)


def build_transformer(frame, feature_columns):
    numeric_columns = [c for c in feature_columns if pd.api.types.is_numeric_dtype(frame[c])]
    categorical_columns = [c for c in feature_columns if c not in numeric_columns]

    numeric_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median"))
    ])

    categorical_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1))
    ])

    transformer = ColumnTransformer(
        transformers=[
            ("num", numeric_pipe, numeric_columns),
            ("cat", categorical_pipe, categorical_columns)
        ],
        remainder="drop"
    )

    return transformer


class DenseModel(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.body = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.20),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.15),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        return self.body(x)


def evaluate_model(model, x_data, y_data, device, batch_size):
    criterion = nn.BCEWithLogitsLoss()
    losses = []
    probs_all = []

    model.eval()
    with torch.no_grad():
        for start in range(0, len(x_data), batch_size):
            xb = torch.from_numpy(x_data[start:start + batch_size]).float().to(device)
            yb = torch.from_numpy(y_data[start:start + batch_size]).float().view(-1, 1).to(device)
            logits = model(xb)
            loss = criterion(logits, yb)
            probs = torch.sigmoid(logits).detach().cpu().numpy().reshape(-1)
            losses.append(loss.item() * len(xb))
            probs_all.append(probs)

    probs_all = np.concatenate(probs_all, axis=0)
    preds = (probs_all >= 0.5).astype(int)
    avg_loss = float(np.sum(losses) / len(x_data))

    return avg_loss, probs_all, preds


def fit_model(x_train, y_train, x_valid, y_valid, device, config):
    model = DenseModel(x_train.shape[1]).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=config["learning_rate"], weight_decay=config["weight_decay"])
    criterion = nn.BCEWithLogitsLoss()

    best_state = copy.deepcopy(model.state_dict())
    best_loss = np.inf
    wait = 0
    indices = np.arange(len(x_train))

    for _ in range(config["epochs"]):
        np.random.shuffle(indices)
        model.train()

        for start in range(0, len(indices), config["batch_size"]):
            idx = indices[start:start + config["batch_size"]]
            xb = torch.from_numpy(x_train[idx]).float().to(device)
            yb = torch.from_numpy(y_train[idx]).float().view(-1, 1).to(device)

            optimizer.zero_grad(set_to_none=True)
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()

        valid_loss, _, _ = evaluate_model(model, x_valid, y_valid, device=device, batch_size=config["batch_size"])

        if valid_loss < best_loss - 1e-6:
            best_loss = valid_loss
            best_state = copy.deepcopy(model.state_dict())
            wait = 0
        else:
            wait += 1
            if wait >= config["patience"]:
                break

    model.load_state_dict(best_state)
    model.eval()
    return model


def gaussian_corruption(x_data, sigma, clip_min, clip_max, seed=42):
    rng = np.random.default_rng(seed)
    noise = rng.normal(0.0, sigma, size=x_data.shape).astype(np.float32)
    corrupted = x_data + noise
    corrupted = np.clip(corrupted, clip_min, clip_max)
    return corrupted.astype(np.float32)


def metric_bundle(y_true, y_prob, threshold=0.5):
    y_pred = (y_prob >= threshold).astype(int)
    acc = accuracy_score(y_true, y_pred)
    bacc = balanced_accuracy_score(y_true, y_pred)
    macro_precision = precision_score(y_true, y_pred, average="macro", zero_division=0)
    macro_recall = recall_score(y_true, y_pred, average="macro", zero_division=0)
    macro_f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
    attack_precision = precision_score(y_true, y_pred, pos_label=1, zero_division=0)
    attack_recall = recall_score(y_true, y_pred, pos_label=1, zero_division=0)
    attack_f1 = f1_score(y_true, y_pred, pos_label=1, zero_division=0)
    benign_precision = precision_score(y_true, y_pred, pos_label=0, zero_division=0)
    benign_recall = recall_score(y_true, y_pred, pos_label=0, zero_division=0)
    benign_f1 = f1_score(y_true, y_pred, pos_label=0, zero_division=0)
    try:
        roc_auc = roc_auc_score(y_true, y_prob)
    except Exception:
        roc_auc = np.nan
    try:
        pr_auc = average_precision_score(y_true, y_prob)
    except Exception:
        pr_auc = np.nan

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    fpr = fp / (fp + tn) if (fp + tn) > 0 else np.nan
    miscls = fp + fn

    return {
        "accuracy": acc,
        "balanced_accuracy": bacc,
        "macro_precision": macro_precision,
        "macro_recall": macro_recall,
        "macro_f1": macro_f1,
        "attack_precision": attack_precision,
        "attack_recall": attack_recall,
        "attack_f1": attack_f1,
        "benign_precision": benign_precision,
        "benign_recall": benign_recall,
        "benign_f1": benign_f1,
        "roc_auc": roc_auc,
        "pr_auc": pr_auc,
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
        "false_positive_rate": fpr,
        "misclassifications": int(miscls)
    }


def prepare_frame(dataset_key, frame):
    frame = standardize_columns(frame)
    frame = coerce_basic_types(frame)
    label_col, family_col = derive_label_columns(dataset_key, frame)

    work_frame = frame.copy()
    work_frame["raw_attack_label"] = work_frame[label_col].astype(str).map(normalize_text)

    if family_col is not None and family_col in work_frame.columns:
        family_values = work_frame[family_col]
    else:
        family_values = pd.Series([""] * len(work_frame), index=work_frame.index)

    work_frame["family_label"] = [
        derive_family(dataset_key, raw_val, fam_val)
        for raw_val, fam_val in zip(work_frame["raw_attack_label"], family_values)
    ]
    work_frame["binary_label"] = (work_frame["family_label"] != "normal").astype(int)
    work_frame["dataset_id"] = dataset_key

    protected = {"source_file", "raw_attack_label", "family_label", "binary_label", "dataset_id"}
    work_frame = drop_redundant_columns(work_frame, protected)

    return work_frame


def make_train_valid_split(frame, valid_fraction, random_state):
    feature_columns = [c for c in frame.columns if c not in {"source_file", "raw_attack_label", "family_label", "binary_label", "dataset_id"}]
    frame = frame.copy()
    frame["record_hash"] = stable_record_hash(frame, feature_columns)
    frame = frame.drop_duplicates(subset=["record_hash"]).reset_index(drop=True)

    feature_columns = [c for c in frame.columns if c not in {"source_file", "raw_attack_label", "family_label", "binary_label", "dataset_id", "record_hash"}]
    frame["stratify_label"] = frame["family_label"].astype(str)
    counts = frame["stratify_label"].value_counts()
    rare = set(counts[counts < 3].index.tolist())
    frame["stratify_label"] = frame["stratify_label"].apply(lambda x: "rare_attack" if x in rare and x != "normal" else x)

    splitter = StratifiedGroupShuffleSplit(
        n_splits=1,
        test_size=valid_fraction,
        random_state=random_state
    )
    train_idx, valid_idx = next(splitter.split(frame, frame["stratify_label"], groups=frame["record_hash"]))

    train_frame = frame.iloc[train_idx].copy().reset_index(drop=True)
    valid_frame = frame.iloc[valid_idx].copy().reset_index(drop=True)

    return train_frame, valid_frame


def run_single_dataset(dataset_key, train_frame, valid_frame, test_frame):
    keep_columns = {"source_file", "raw_attack_label", "family_label", "binary_label", "dataset_id", "record_hash", "stratify_label"}
    feature_columns = [c for c in train_frame.columns if c not in keep_columns]
    feature_columns = [c for c in feature_columns if c in test_frame.columns]
    feature_columns = [c for c in feature_columns if c in valid_frame.columns]

    transformer = build_transformer(train_frame, feature_columns)

    x_train_base = transformer.fit_transform(train_frame[feature_columns])
    x_valid_base = transformer.transform(valid_frame[feature_columns])
    x_test_base = transformer.transform(test_frame[feature_columns])

    scaler = StandardScaler()
    x_train = scaler.fit_transform(x_train_base).astype(np.float32)
    x_valid = scaler.transform(x_valid_base).astype(np.float32)
    x_test = scaler.transform(x_test_base).astype(np.float32)

    y_train = train_frame["binary_label"].astype(int).values
    y_valid = valid_frame["binary_label"].astype(int).values
    y_test = test_frame["binary_label"].astype(int).values

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = fit_model(x_train, y_train, x_valid, y_valid, device=device, config=train_config)

    _, clean_prob, clean_pred = evaluate_model(model, x_test, y_test, device=device, batch_size=train_config["batch_size"])

    clip_min = x_train.min(axis=0).astype(np.float32)
    clip_max = x_train.max(axis=0).astype(np.float32)
    x_test_gaussian = gaussian_corruption(x_test, sigma=gaussian_config["sigma"], clip_min=clip_min, clip_max=clip_max, seed=42)

    _, gaussian_prob, gaussian_pred = evaluate_model(model, x_test_gaussian, y_test, device=device, batch_size=train_config["batch_size"])

    return {
        "dataset_id": dataset_key,
        "feature_columns": feature_columns,
        "y_test": y_test,
        "clean_prob": clean_prob,
        "clean_pred": clean_pred,
        "gaussian_prob": gaussian_prob,
        "gaussian_pred": gaussian_pred
    }


nsl_train_raw = load_dataset("nsl_kdd", data_sources["nsl_kdd"]["train"])
nsl_test_raw = load_dataset("nsl_kdd", data_sources["nsl_kdd"]["test"])

nsl_train_frame = prepare_frame("nsl_kdd", nsl_train_raw)
nsl_test_frame = prepare_frame("nsl_kdd", nsl_test_raw)

shared_columns = sorted(set(nsl_train_frame.columns).intersection(set(nsl_test_frame.columns)))
nsl_train_frame = nsl_train_frame[shared_columns].copy()
nsl_test_frame = nsl_test_frame[shared_columns].copy()

train_frame, valid_frame = make_train_valid_split(
    nsl_train_frame,
    valid_fraction=split_config["valid_fraction"],
    random_state=split_config["random_state"]
)

nsl_feature_columns = [c for c in train_frame.columns if c not in {"source_file", "raw_attack_label", "family_label", "binary_label", "dataset_id", "record_hash", "stratify_label"}]
nsl_test_frame = nsl_test_frame.copy()
nsl_test_frame["record_hash"] = stable_record_hash(nsl_test_frame, [c for c in nsl_feature_columns if c in nsl_test_frame.columns])
nsl_test_frame = nsl_test_frame.drop_duplicates(subset=["record_hash"]).reset_index(drop=True)
nsl_test_frame["stratify_label"] = nsl_test_frame["family_label"].astype(str)

nsl_result = run_single_dataset("nsl_kdd", train_frame, valid_frame, nsl_test_frame)

clean_metrics = metric_bundle(nsl_result["y_test"], nsl_result["clean_prob"])
class_precision, class_recall, class_f1, class_support = precision_recall_fscore_support(
    nsl_result["y_test"],
    nsl_result["clean_pred"],
    labels=[0, 1],
    zero_division=0
)

metrics_rows = [
    {
        "row_type": "overall",
        "class_label": "overall",
        "sample_count": int(len(nsl_result["y_test"])),
        "support": int(len(nsl_result["y_test"])),
        "accuracy": clean_metrics["accuracy"],
        "balanced_accuracy": clean_metrics["balanced_accuracy"],
        "macro_precision": clean_metrics["macro_precision"],
        "macro_recall": clean_metrics["macro_recall"],
        "macro_f1": clean_metrics["macro_f1"],
        "precision": np.nan,
        "recall": np.nan,
        "f1": np.nan,
        "roc_auc": clean_metrics["roc_auc"],
        "pr_auc": clean_metrics["pr_auc"],
        "tn": clean_metrics["tn"],
        "fp": clean_metrics["fp"],
        "fn": clean_metrics["fn"],
        "tp": clean_metrics["tp"],
        "false_positive_rate": clean_metrics["false_positive_rate"],
        "misclassifications": clean_metrics["misclassifications"]
    },
    {
        "row_type": "class",
        "class_label": "benign",
        "sample_count": int(len(nsl_result["y_test"])),
        "support": int(class_support[0]),
        "accuracy": np.nan,
        "balanced_accuracy": np.nan,
        "macro_precision": np.nan,
        "macro_recall": np.nan,
        "macro_f1": np.nan,
        "precision": class_precision[0],
        "recall": class_recall[0],
        "f1": class_f1[0],
        "roc_auc": np.nan,
        "pr_auc": np.nan,
        "tn": np.nan,
        "fp": np.nan,
        "fn": np.nan,
        "tp": np.nan,
        "false_positive_rate": np.nan,
        "misclassifications": np.nan
    },
    {
        "row_type": "class",
        "class_label": "attack",
        "sample_count": int(len(nsl_result["y_test"])),
        "support": int(class_support[1]),
        "accuracy": np.nan,
        "balanced_accuracy": np.nan,
        "macro_precision": np.nan,
        "macro_recall": np.nan,
        "macro_f1": np.nan,
        "precision": class_precision[1],
        "recall": class_recall[1],
        "f1": class_f1[1],
        "roc_auc": np.nan,
        "pr_auc": np.nan,
        "tn": np.nan,
        "fp": np.nan,
        "fn": np.nan,
        "tp": np.nan,
        "false_positive_rate": np.nan,
        "misclassifications": np.nan
    }
]

metrics_frame = pd.DataFrame(metrics_rows)
metrics_frame.to_csv(output_dir / "phase4_nslkdd_metrics.csv", index=False)

relevance_rows = [
    {
        "metric_name": "Accuracy",
        "operational_role": "Overall correctness on the hold-out stream",
        "cloud_ids_relevance": "Useful as a high-level summary but insufficient alone when benign and attack traffic are imbalanced."
    },
    {
        "metric_name": "Balanced Accuracy",
        "operational_role": "Average sensitivity across benign and attack classes",
        "cloud_ids_relevance": "Shows whether performance remains stable when one class dominates operational traffic."
    },
    {
        "metric_name": "Macro Precision",
        "operational_role": "Mean precision across both classes",
        "cloud_ids_relevance": "Penalizes false positives on benign traffic and overconfident attack assignments."
    },
    {
        "metric_name": "Macro Recall",
        "operational_role": "Mean recall across both classes",
        "cloud_ids_relevance": "Reflects whether attacks are missed while benign decisions also remain recoverable."
    },
    {
        "metric_name": "Macro F1",
        "operational_role": "Balanced harmonic view of precision and recall",
        "cloud_ids_relevance": "Useful for deployment-facing comparison when both missed attacks and false alarms matter."
    },
    {
        "metric_name": "ROC-AUC",
        "operational_role": "Threshold-independent ranking quality",
        "cloud_ids_relevance": "Shows separability of benign and attack samples before a fixed alarm threshold is chosen."
    },
    {
        "metric_name": "PR-AUC",
        "operational_role": "Precision-recall trade-off over thresholds",
        "cloud_ids_relevance": "More informative than ROC-AUC when attack prevalence is low or operational imbalance is strong."
    },
    {
        "metric_name": "False Positive Rate",
        "operational_role": "Fraction of benign events incorrectly escalated",
        "cloud_ids_relevance": "Directly tied to analyst burden, alert fatigue, and unnecessary operational disruption."
    },
    {
        "metric_name": "True Positives and False Negatives",
        "operational_role": "Detected and missed intrusions",
        "cloud_ids_relevance": "These counts expose how many actual attacks are captured versus silently missed."
    },
    {
        "metric_name": "True Negatives and False Positives",
        "operational_role": "Correctly cleared and incorrectly flagged benign events",
        "cloud_ids_relevance": "These counts show whether routine cloud activity remains usable without excessive alarm volume."
    }
]

relevance_frame = pd.DataFrame(relevance_rows)
relevance_frame.to_csv(output_dir / "phase4_metric_relevance_table.csv", index=False)

comparison_rows = []

comparison_rows.append({
    "dataset_id": "nsl_kdd",
    "condition": "clean",
    "sample_count": int(len(nsl_result["y_test"])),
    "benign_count": int((nsl_result["y_test"] == 0).sum()),
    "attack_count": int((nsl_result["y_test"] == 1).sum()),
    "accuracy": clean_metrics["accuracy"],
    "balanced_accuracy": clean_metrics["balanced_accuracy"],
    "macro_f1": clean_metrics["macro_f1"],
    "attack_precision": clean_metrics["attack_precision"],
    "attack_recall": clean_metrics["attack_recall"],
    "attack_f1": clean_metrics["attack_f1"],
    "roc_auc": clean_metrics["roc_auc"],
    "pr_auc": clean_metrics["pr_auc"],
    "fp": clean_metrics["fp"],
    "false_positive_rate": clean_metrics["false_positive_rate"],
    "misclassifications": clean_metrics["misclassifications"],
    "gaussian_sigma": 0.0
})

gaussian_metrics_nsl = metric_bundle(nsl_result["y_test"], nsl_result["gaussian_prob"])
comparison_rows.append({
    "dataset_id": "nsl_kdd",
    "condition": "gaussian",
    "sample_count": int(len(nsl_result["y_test"])),
    "benign_count": int((nsl_result["y_test"] == 0).sum()),
    "attack_count": int((nsl_result["y_test"] == 1).sum()),
    "accuracy": gaussian_metrics_nsl["accuracy"],
    "balanced_accuracy": gaussian_metrics_nsl["balanced_accuracy"],
    "macro_f1": gaussian_metrics_nsl["macro_f1"],
    "attack_precision": gaussian_metrics_nsl["attack_precision"],
    "attack_recall": gaussian_metrics_nsl["attack_recall"],
    "attack_f1": gaussian_metrics_nsl["attack_f1"],
    "roc_auc": gaussian_metrics_nsl["roc_auc"],
    "pr_auc": gaussian_metrics_nsl["pr_auc"],
    "fp": gaussian_metrics_nsl["fp"],
    "false_positive_rate": gaussian_metrics_nsl["false_positive_rate"],
    "misclassifications": gaussian_metrics_nsl["misclassifications"],
    "gaussian_sigma": gaussian_config["sigma"]
})

for dataset_key in ["cicids2017", "unsw_nb15"]:
    raw_frame = load_dataset(dataset_key, data_sources[dataset_key])
    prepared_frame = prepare_frame(dataset_key, raw_frame)
    prepared_frame["record_hash"] = stable_record_hash(
        prepared_frame,
        [c for c in prepared_frame.columns if c not in {"source_file", "raw_attack_label", "family_label", "binary_label", "dataset_id"}]
    )
    prepared_frame = prepared_frame.drop_duplicates(subset=["record_hash"]).reset_index(drop=True)
    prepared_frame["stratify_label"] = prepared_frame["family_label"].astype(str)
    family_counts = prepared_frame["stratify_label"].value_counts()
    rare = set(family_counts[family_counts < 3].index.tolist())
    prepared_frame["stratify_label"] = prepared_frame["stratify_label"].apply(lambda x: "rare_attack" if x in rare and x != "normal" else x)

    splitter_outer = StratifiedGroupShuffleSplit(
        n_splits=1,
        test_size=0.15,
        random_state=42
    )
    remainder_idx, test_idx = next(
        splitter_outer.split(prepared_frame, prepared_frame["stratify_label"], groups=prepared_frame["record_hash"])
    )
    remainder_frame = prepared_frame.iloc[remainder_idx].copy().reset_index(drop=True)
    test_frame_ds = prepared_frame.iloc[test_idx].copy().reset_index(drop=True)

    train_frame_ds, valid_frame_ds = make_train_valid_split(
        remainder_frame.drop(columns=[]),
        valid_fraction=split_config["valid_fraction"],
        random_state=split_config["random_state"]
    )

    ds_result = run_single_dataset(dataset_key, train_frame_ds, valid_frame_ds, test_frame_ds)

    ds_clean = metric_bundle(ds_result["y_test"], ds_result["clean_prob"])
    ds_gaussian = metric_bundle(ds_result["y_test"], ds_result["gaussian_prob"])

    comparison_rows.append({
        "dataset_id": dataset_key,
        "condition": "clean",
        "sample_count": int(len(ds_result["y_test"])),
        "benign_count": int((ds_result["y_test"] == 0).sum()),
        "attack_count": int((ds_result["y_test"] == 1).sum()),
        "accuracy": ds_clean["accuracy"],
        "balanced_accuracy": ds_clean["balanced_accuracy"],
        "macro_f1": ds_clean["macro_f1"],
        "attack_precision": ds_clean["attack_precision"],
        "attack_recall": ds_clean["attack_recall"],
        "attack_f1": ds_clean["attack_f1"],
        "roc_auc": ds_clean["roc_auc"],
        "pr_auc": ds_clean["pr_auc"],
        "fp": ds_clean["fp"],
        "false_positive_rate": ds_clean["false_positive_rate"],
        "misclassifications": ds_clean["misclassifications"],
        "gaussian_sigma": 0.0
    })

    comparison_rows.append({
        "dataset_id": dataset_key,
        "condition": "gaussian",
        "sample_count": int(len(ds_result["y_test"])),
        "benign_count": int((ds_result["y_test"] == 0).sum()),
        "attack_count": int((ds_result["y_test"] == 1).sum()),
        "accuracy": ds_gaussian["accuracy"],
        "balanced_accuracy": ds_gaussian["balanced_accuracy"],
        "macro_f1": ds_gaussian["macro_f1"],
        "attack_precision": ds_gaussian["attack_precision"],
        "attack_recall": ds_gaussian["attack_recall"],
        "attack_f1": ds_gaussian["attack_f1"],
        "roc_auc": ds_gaussian["roc_auc"],
        "pr_auc": ds_gaussian["pr_auc"],
        "fp": ds_gaussian["fp"],
        "false_positive_rate": ds_gaussian["false_positive_rate"],
        "misclassifications": ds_gaussian["misclassifications"],
        "gaussian_sigma": gaussian_config["sigma"]
    })

comparison_frame = pd.DataFrame(comparison_rows).sort_values(["dataset_id", "condition"]).reset_index(drop=True)
comparison_frame.to_csv(output_dir / "phase4_false_alarm_gaussian_baseline.csv", index=False)

heatmap_values = np.array([
    [class_precision[0], class_recall[0], class_f1[0]],
    [class_precision[1], class_recall[1], class_f1[1]]
])

plt.figure(figsize=(6, 4))
plt.imshow(heatmap_values, aspect="auto")
plt.xticks([0, 1, 2], ["Precision", "Recall", "F1"])
plt.yticks([0, 1], ["Benign", "Attack"])
for i in range(heatmap_values.shape[0]):
    for j in range(heatmap_values.shape[1]):
        plt.text(j, i, f"{heatmap_values[i, j]:.3f}", ha="center", va="center")
plt.colorbar()
plt.tight_layout()
plt.savefig(output_dir / "phase4_class_metric_heatmap.png", dpi=300, bbox_inches="tight")
plt.close()

conf_mat = confusion_matrix(nsl_result["y_test"], nsl_result["clean_pred"], labels=[0, 1])

plt.figure(figsize=(5, 4))
plt.imshow(conf_mat, aspect="auto")
plt.xticks([0, 1], ["Predicted Benign", "Predicted Attack"], rotation=15)
plt.yticks([0, 1], ["Actual Benign", "Actual Attack"])
for i in range(conf_mat.shape[0]):
    for j in range(conf_mat.shape[1]):
        plt.text(j, i, str(conf_mat[i, j]), ha="center", va="center")
plt.colorbar()
plt.tight_layout()
plt.savefig(output_dir / "phase4_confusion_matrix.png", dpi=300, bbox_inches="tight")
plt.close()

fpr_vals, tpr_vals, _ = roc_curve(nsl_result["y_test"], nsl_result["clean_prob"])
roc_area = roc_auc_score(nsl_result["y_test"], nsl_result["clean_prob"])

plt.figure(figsize=(6, 5))
plt.plot(fpr_vals, tpr_vals, label=f"AUC = {roc_area:.4f}")
plt.plot([0, 1], [0, 1], linestyle="--")
plt.xlim(0, 1)
plt.ylim(0, 1)
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend()
plt.tight_layout()
plt.savefig(output_dir / "phase4_roc_curve.png", dpi=300, bbox_inches="tight")
plt.close()

In [ ]:
from pathlib import Path
import copy
import hashlib
import io
import json
import math
import os
import pickle
import random
import re
import time
import warnings
from datetime import datetime

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, balanced_accuracy_score, confusion_matrix, f1_score, precision_recall_fscore_support, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import StratifiedGroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder, StandardScaler

warnings.filterwarnings("ignore")

try:
    import psutil
except Exception:
    psutil = None

try:
    import resource
except Exception:
    resource = None

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

output_dir = Path("/kaggle/working/results/phase5")
output_dir.mkdir(parents=True, exist_ok=True)

data_sources = {
    "nsl_kdd_train": [
        "/kaggle/input/nsl-kdd/KDDTrain+.txt"
    ],
    "nsl_kdd_test": [
        "/kaggle/input/nsl-kdd/KDDTest+.txt"
    ]
}

split_config = {
    "valid_fraction": 0.15,
    "random_state": 42
}

train_config = {
    "epochs": 40,
    "batch_size": 2048,
    "learning_rate": 1e-3,
    "weight_decay": 1e-4,
    "patience": 8
}

ledger_config = {
    "live_node_count": 4,
    "benchmark_node_counts": [1, 4],
    "live_stream_blocks": 1000,
    "sample_export_blocks": 250,
    "throughput_window": 50
}

cost_config = {
    "hours_per_month": 730,
    "vcpu_hour_rate_usd": 0.05,
    "memory_gb_hour_rate_usd": 0.007,
    "storage_gb_month_rate_usd": 0.02,
    "artifact_storage_gb_month_rate_usd": 0.02
}

nsl_columns = [
    "duration", "protocol_type", "service", "flag", "src_bytes", "dst_bytes", "land",
    "wrong_fragment", "urgent", "hot", "num_failed_logins", "logged_in", "num_compromised",
    "root_shell", "su_attempted", "num_root", "num_file_creations", "num_shells",
    "num_access_files", "num_outbound_cmds", "is_host_login", "is_guest_login", "count",
    "srv_count", "serror_rate", "srv_serror_rate", "rerror_rate", "srv_rerror_rate",
    "same_srv_rate", "diff_srv_rate", "srv_diff_host_rate", "dst_host_count",
    "dst_host_srv_count", "dst_host_same_srv_rate", "dst_host_diff_srv_rate",
    "dst_host_same_src_port_rate", "dst_host_srv_diff_host_rate", "dst_host_serror_rate",
    "dst_host_srv_serror_rate", "dst_host_rerror_rate", "dst_host_srv_rerror_rate",
    "raw_label", "difficulty"
]

nsl_family_map = {
    "normal": "normal",
    "back": "dos",
    "land": "dos",
    "neptune": "dos",
    "pod": "dos",
    "smurf": "dos",
    "teardrop": "dos",
    "mailbomb": "dos",
    "apache2": "dos",
    "processtable": "dos",
    "udpstorm": "dos",
    "ipsweep": "probe",
    "nmap": "probe",
    "portsweep": "probe",
    "satan": "probe",
    "mscan": "probe",
    "saint": "probe",
    "ftp_write": "r2l",
    "guess_passwd": "r2l",
    "imap": "r2l",
    "multihop": "r2l",
    "phf": "r2l",
    "spy": "r2l",
    "warezclient": "r2l",
    "warezmaster": "r2l",
    "sendmail": "r2l",
    "named": "r2l",
    "snmpgetattack": "r2l",
    "snmpguess": "r2l",
    "xlock": "r2l",
    "xsnoop": "r2l",
    "worm": "r2l",
    "buffer_overflow": "u2r",
    "loadmodule": "u2r",
    "perl": "u2r",
    "rootkit": "u2r",
    "httptunnel": "u2r",
    "ps": "u2r",
    "sqlattack": "u2r",
    "xterm": "u2r"
}

attack_label_candidates = [
    "label", "labels", "class", "target", "attack", "attack_type", "attack_cat",
    "category", "traffic_label", "raw_label", "rawlabel"
]

family_label_candidates = [
    "attack_family", "family", "attack_cat", "attack_category", "category_group"
]

drop_candidates = {
    "flow_id", "flowid", "src_ip", "dst_ip", "srcip", "dstip", "timestamp", "time", "id",
    "index", "idx", "session_id", "record_id"
}


def normalize_name(value):
    value = str(value).strip().lower()
    value = re.sub(r"[\s\-/]+", "_", value)
    value = re.sub(r"[^a-z0-9_]", "", value)
    value = re.sub(r"_+", "_", value).strip("_")
    return value


def normalize_text(value):
    if pd.isna(value):
        return ""
    value = str(value).strip().lower()
    value = value.replace("\x00", "")
    value = re.sub(r"\s+", " ", value)
    return value


def resolve_files(entries):
    files = []
    for entry in entries:
        path_obj = Path(entry)
        if path_obj.is_dir():
            files.extend(sorted([p for p in path_obj.rglob("*") if p.suffix.lower() in {".csv", ".txt", ".data", ".parquet"}]))
        elif path_obj.exists():
            files.append(path_obj)
    return files


def read_single_file(path_obj):
    suffix = path_obj.suffix.lower()
    if suffix == ".parquet":
        return pd.read_parquet(path_obj)
    try:
        frame = pd.read_csv(path_obj, low_memory=False)
        if frame.shape[1] > 1:
            return frame
    except Exception:
        pass
    try:
        frame = pd.read_csv(path_obj, header=None, low_memory=False)
        if frame.shape[1] > 1:
            return frame
    except Exception:
        pass
    try:
        return pd.read_csv(path_obj, sep=None, engine="python", low_memory=False)
    except Exception:
        return pd.read_csv(path_obj, sep=r"\s+", engine="python", low_memory=False)


def standardize_columns(frame):
    cols = [normalize_name(c) for c in frame.columns]
    used = {}
    final_cols = []
    for col in cols:
        if col not in used:
            used[col] = 0
            final_cols.append(col)
        else:
            used[col] += 1
            final_cols.append(f"{col}_{used[col]}")
    frame.columns = final_cols
    return frame


def assign_known_schema(dataset_key, frame):
    if dataset_key == "nsl_kdd":
        if frame.shape[1] == len(nsl_columns):
            frame.columns = nsl_columns
        elif frame.shape[1] == len(nsl_columns) - 1:
            frame.columns = nsl_columns[:-1]
            frame["difficulty"] = 0
    return frame


def load_dataset(dataset_key, entries):
    parts = []
    for path_obj in resolve_files(entries):
        frame = read_single_file(path_obj)
        frame = assign_known_schema(dataset_key, frame)
        frame = standardize_columns(frame)
        frame["source_file"] = path_obj.name
        parts.append(frame)
    if not parts:
        raise FileNotFoundError(dataset_key)
    frame = pd.concat(parts, axis=0, ignore_index=True)
    frame = standardize_columns(frame)
    return frame


def select_first_present(columns, candidates):
    lookup = {normalize_name(c): c for c in columns}
    for candidate in candidates:
        key = normalize_name(candidate)
        if key in lookup:
            return lookup[key]
    return None


def derive_label_columns(dataset_key, frame):
    label_col = select_first_present(frame.columns, attack_label_candidates)
    family_col = select_first_present(frame.columns, family_label_candidates)

    if dataset_key == "nsl_kdd":
        if "raw_label" in frame.columns:
            label_col = "raw_label"
        family_col = None

    if label_col is None:
        raise ValueError(dataset_key)
    return label_col, family_col


def coerce_basic_types(frame):
    frame = frame.copy()
    frame = frame.replace([np.inf, -np.inf], np.nan)
    for col in frame.columns:
        if frame[col].dtype == object:
            converted = pd.to_numeric(frame[col], errors="coerce")
            if converted.notna().mean() >= 0.90:
                frame[col] = converted
    entirely_missing = [c for c in frame.columns if frame[c].isna().all()]
    if entirely_missing:
        frame = frame.drop(columns=entirely_missing)
    return frame


def drop_redundant_columns(frame, protected):
    columns_to_drop = []
    for col in frame.columns:
        if col in protected:
            continue
        if normalize_name(col) in drop_candidates:
            columns_to_drop.append(col)
            continue
        if frame[col].nunique(dropna=False) <= 1:
            columns_to_drop.append(col)
    if columns_to_drop:
        frame = frame.drop(columns=columns_to_drop)
    return frame


def nsl_family_from_label(raw_value):
    raw_value = normalize_text(raw_value).rstrip(".")
    return nsl_family_map.get(raw_value, "other_attack")


def stable_record_hash(frame, feature_columns):
    hash_frame = frame[feature_columns].copy()
    for col in hash_frame.columns:
        if pd.api.types.is_numeric_dtype(hash_frame[col]):
            hash_frame[col] = hash_frame[col].astype("float64").round(8)
        else:
            hash_frame[col] = hash_frame[col].astype(str).str.strip().str.lower()
    return pd.util.hash_pandas_object(hash_frame, index=False).astype(str)


def build_transformer(frame, feature_columns):
    numeric_columns = [c for c in feature_columns if pd.api.types.is_numeric_dtype(frame[c])]
    categorical_columns = [c for c in feature_columns if c not in numeric_columns]

    numeric_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median"))
    ])

    categorical_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1))
    ])

    transformer = ColumnTransformer(
        transformers=[
            ("num", numeric_pipe, numeric_columns),
            ("cat", categorical_pipe, categorical_columns)
        ],
        remainder="drop"
    )

    return transformer


class DenseModel(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.body = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.20),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.15),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        return self.body(x)


def evaluate_model(model, x_data, y_data, device, batch_size):
    criterion = nn.BCEWithLogitsLoss()
    losses = []
    probs_all = []

    model.eval()
    with torch.no_grad():
        for start in range(0, len(x_data), batch_size):
            xb = torch.from_numpy(x_data[start:start + batch_size]).float().to(device)
            yb = torch.from_numpy(y_data[start:start + batch_size]).float().view(-1, 1).to(device)
            logits = model(xb)
            loss = criterion(logits, yb)
            probs = torch.sigmoid(logits).detach().cpu().numpy().reshape(-1)
            losses.append(loss.item() * len(xb))
            probs_all.append(probs)

    probs_all = np.concatenate(probs_all, axis=0)
    preds = (probs_all >= 0.5).astype(int)
    avg_loss = float(np.sum(losses) / len(x_data))
    return avg_loss, probs_all, preds


def fit_model(x_train, y_train, x_valid, y_valid, device, config):
    model = DenseModel(x_train.shape[1]).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=config["learning_rate"], weight_decay=config["weight_decay"])
    criterion = nn.BCEWithLogitsLoss()

    best_state = copy.deepcopy(model.state_dict())
    best_loss = np.inf
    wait = 0
    indices = np.arange(len(x_train))

    for _ in range(config["epochs"]):
        np.random.shuffle(indices)
        model.train()

        for start in range(0, len(indices), config["batch_size"]):
            idx = indices[start:start + config["batch_size"]]
            xb = torch.from_numpy(x_train[idx]).float().to(device)
            yb = torch.from_numpy(y_train[idx]).float().view(-1, 1).to(device)

            optimizer.zero_grad(set_to_none=True)
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()

        valid_loss, _, _ = evaluate_model(model, x_valid, y_valid, device=device, batch_size=config["batch_size"])

        if valid_loss < best_loss - 1e-6:
            best_loss = valid_loss
            best_state = copy.deepcopy(model.state_dict())
            wait = 0
        else:
            wait += 1
            if wait >= config["patience"]:
                break

    model.load_state_dict(best_state)
    model.eval()
    return model


def prepare_frame(dataset_key, frame):
    frame = standardize_columns(frame)
    frame = coerce_basic_types(frame)
    label_col, family_col = derive_label_columns(dataset_key, frame)

    work_frame = frame.copy()
    work_frame["raw_attack_label"] = work_frame[label_col].astype(str).map(normalize_text)

    if family_col is not None and family_col in work_frame.columns:
        family_values = work_frame[family_col]
    else:
        family_values = pd.Series([""] * len(work_frame), index=work_frame.index)

    work_frame["family_label"] = [
        nsl_family_from_label(raw_val) for raw_val in work_frame["raw_attack_label"]
    ]
    work_frame["binary_label"] = (work_frame["family_label"] != "normal").astype(int)
    work_frame["dataset_id"] = dataset_key

    protected = {"source_file", "raw_attack_label", "family_label", "binary_label", "dataset_id"}
    work_frame = drop_redundant_columns(work_frame, protected)
    return work_frame


def make_train_valid_split(frame, valid_fraction, random_state):
    feature_columns = [c for c in frame.columns if c not in {"source_file", "raw_attack_label", "family_label", "binary_label", "dataset_id"}]
    frame = frame.copy()
    frame["record_hash"] = stable_record_hash(frame, feature_columns)
    frame = frame.drop_duplicates(subset=["record_hash"]).reset_index(drop=True)

    feature_columns = [c for c in frame.columns if c not in {"source_file", "raw_attack_label", "family_label", "binary_label", "dataset_id", "record_hash"}]
    frame["stratify_label"] = frame["family_label"].astype(str)
    counts = frame["stratify_label"].value_counts()
    rare = set(counts[counts < 3].index.tolist())
    frame["stratify_label"] = frame["stratify_label"].apply(lambda x: "rare_attack" if x in rare and x != "normal" else x)

    splitter = StratifiedGroupShuffleSplit(
        n_splits=1,
        test_size=valid_fraction,
        random_state=random_state
    )
    train_idx, valid_idx = next(splitter.split(frame, frame["stratify_label"], groups=frame["record_hash"]))

    train_frame = frame.iloc[train_idx].copy().reset_index(drop=True)
    valid_frame = frame.iloc[valid_idx].copy().reset_index(drop=True)
    return train_frame, valid_frame


def canonical_json(value):
    return json.dumps(value, sort_keys=True, separators=(",", ":"), ensure_ascii=False)


def sha256_hex(value):
    return hashlib.sha256(value.encode("utf-8")).hexdigest()


class LedgerNode:
    def __init__(self, node_name, genesis_hash):
        self.node_name = node_name
        self.genesis_hash = genesis_hash
        self.blocks = []

    def current_hash(self):
        if len(self.blocks) == 0:
            return self.genesis_hash
        return self.blocks[-1]["block_hash"]

    def append_block(self, block):
        if block["block_index"] != len(self.blocks):
            raise ValueError(self.node_name)
        if block["previous_hash"] != self.current_hash():
            raise ValueError(self.node_name)
        self.blocks.append(copy.deepcopy(block))

    def verify_chain(self):
        previous_hash = self.genesis_hash
        for idx, block in enumerate(self.blocks):
            if block["block_index"] != idx:
                return False, idx
            if block["previous_hash"] != previous_hash:
                return False, idx
            block_for_hash = {k: v for k, v in block.items() if k not in {"block_hash", "block_bytes"}}
            expected_hash = sha256_hex(canonical_json(block_for_hash))
            if block["block_hash"] != expected_hash:
                return False, idx
            previous_hash = block["block_hash"]
        return True, -1


class PermissionedLedger:
    def __init__(self, node_count):
        self.node_count = node_count
        self.genesis_hash = sha256_hex(f"GENESIS::{node_count}")
        self.nodes = [LedgerNode(f"node_{idx}", self.genesis_hash) for idx in range(node_count)]

    def build_block(self, payload):
        primary = self.nodes[0]
        block = {
            "block_index": len(primary.blocks),
            "timestamp_utc": datetime.utcnow().isoformat(timespec="milliseconds") + "Z",
            "previous_hash": primary.current_hash(),
            **payload
        }
        block_for_hash = {k: v for k, v in block.items()}
        block_hash = sha256_hex(canonical_json(block_for_hash))
        block["block_hash"] = block_hash
        block["block_bytes"] = len(canonical_json(block).encode("utf-8"))
        return block

    def commit(self, payload):
        overall_start = time.perf_counter()
        block = self.build_block(payload)

        local_start = time.perf_counter()
        self.nodes[0].append_block(block)
        local_append_latency_ms = (time.perf_counter() - local_start) * 1000.0

        for replica in self.nodes[1:]:
            replica.append_block(block)

        replicated_commit_latency_ms = (time.perf_counter() - overall_start) * 1000.0
        return block, local_append_latency_ms, replicated_commit_latency_ms

    def verify_all(self):
        results = []
        for node in self.nodes:
            valid_flag, broken_index = node.verify_chain()
            results.append({
                "node_name": node.node_name,
                "chain_valid": int(valid_flag),
                "broken_index": broken_index
            })
        return results


def get_cpu_time_seconds():
    if psutil is not None:
        process = psutil.Process(os.getpid())
        cpu_times = process.cpu_times()
        return float(cpu_times.user + cpu_times.system)
    return float(time.process_time())


def get_rss_bytes():
    if psutil is not None:
        process = psutil.Process(os.getpid())
        return int(process.memory_info().rss)
    if resource is not None:
        rss_value = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
        if os.name == "posix":
            return int(rss_value * 1024)
        return int(rss_value)
    return 0


def rolling_peak_throughput(latency_ms, window_size):
    arr = np.asarray(latency_ms, dtype=float) / 1000.0
    if len(arr) == 0:
        return np.nan
    window_size = min(window_size, len(arr))
    if window_size == 0:
        return np.nan
    sums = np.convolve(arr, np.ones(window_size), mode="valid")
    return float(np.max(window_size / sums))


def benchmark_ledger(payloads, node_count, throughput_window):
    ledger = PermissionedLedger(node_count=node_count)

    local_latency_ms = []
    commit_latency_ms = []
    block_sizes = []

    rss_peak = get_rss_bytes()
    cpu_start = get_cpu_time_seconds()
    wall_start = time.perf_counter()

    for payload in payloads:
        block, local_ms, commit_ms = ledger.commit(payload)
        local_latency_ms.append(local_ms)
        commit_latency_ms.append(commit_ms)
        block_sizes.append(block["block_bytes"])
        rss_peak = max(rss_peak, get_rss_bytes())

    wall_elapsed = time.perf_counter() - wall_start
    cpu_elapsed = get_cpu_time_seconds() - cpu_start
    cpu_util_pct = (cpu_elapsed / wall_elapsed / max(os.cpu_count(), 1)) * 100.0 if wall_elapsed > 0 else np.nan

    verification_rows = ledger.verify_all()
    chain_valid = int(all(row["chain_valid"] == 1 for row in verification_rows))
    chain_bytes_per_node = float(np.sum(block_sizes)) if len(block_sizes) > 0 else 0.0

    result = {
        "mode_name": f"{node_count}_node" if node_count > 1 else "single_node",
        "node_count": int(node_count),
        "block_count": int(len(payloads)),
        "mean_local_append_latency_ms": float(np.mean(local_latency_ms)) if len(local_latency_ms) > 0 else np.nan,
        "p50_local_append_latency_ms": float(np.percentile(local_latency_ms, 50)) if len(local_latency_ms) > 0 else np.nan,
        "p95_local_append_latency_ms": float(np.percentile(local_latency_ms, 95)) if len(local_latency_ms) > 0 else np.nan,
        "mean_replicated_commit_latency_ms": float(np.mean(commit_latency_ms)) if len(commit_latency_ms) > 0 else np.nan,
        "p50_replicated_commit_latency_ms": float(np.percentile(commit_latency_ms, 50)) if len(commit_latency_ms) > 0 else np.nan,
        "p95_replicated_commit_latency_ms": float(np.percentile(commit_latency_ms, 95)) if len(commit_latency_ms) > 0 else np.nan,
        "sustained_throughput_blocks_per_sec": float(len(payloads) / wall_elapsed) if wall_elapsed > 0 else np.nan,
        "peak_window_throughput_blocks_per_sec": rolling_peak_throughput(commit_latency_ms, throughput_window),
        "mean_block_bytes": float(np.mean(block_sizes)) if len(block_sizes) > 0 else np.nan,
        "max_block_bytes": float(np.max(block_sizes)) if len(block_sizes) > 0 else np.nan,
        "chain_bytes_per_node": chain_bytes_per_node,
        "chain_bytes_all_nodes": chain_bytes_per_node * node_count,
        "cpu_utilization_percent": float(cpu_util_pct),
        "peak_rss_mb": float(rss_peak / (1024 ** 2)),
        "chain_valid": int(chain_valid)
    }

    return result, ledger


nsl_train_raw = load_dataset("nsl_kdd", data_sources["nsl_kdd_train"])
nsl_test_raw = load_dataset("nsl_kdd", data_sources["nsl_kdd_test"])

nsl_train_frame = prepare_frame("nsl_kdd", nsl_train_raw)
nsl_test_frame = prepare_frame("nsl_kdd", nsl_test_raw)

shared_columns = sorted(set(nsl_train_frame.columns).intersection(set(nsl_test_frame.columns)))
nsl_train_frame = nsl_train_frame[shared_columns].copy()
nsl_test_frame = nsl_test_frame[shared_columns].copy()

train_frame, valid_frame = make_train_valid_split(
    nsl_train_frame,
    valid_fraction=split_config["valid_fraction"],
    random_state=split_config["random_state"]
)

nsl_feature_columns = [
    c for c in train_frame.columns
    if c not in {"source_file", "raw_attack_label", "family_label", "binary_label", "dataset_id", "record_hash", "stratify_label"}
]
nsl_test_frame = nsl_test_frame.copy()
nsl_test_frame["record_hash"] = stable_record_hash(nsl_test_frame, [c for c in nsl_feature_columns if c in nsl_test_frame.columns])
nsl_test_frame = nsl_test_frame.drop_duplicates(subset=["record_hash"]).reset_index(drop=True)
nsl_test_frame["stratify_label"] = nsl_test_frame["family_label"].astype(str)
nsl_test_frame["stream_index"] = np.arange(len(nsl_test_frame))

feature_columns = [c for c in train_frame.columns if c in nsl_test_frame.columns and c in valid_frame.columns]
feature_columns = [
    c for c in feature_columns
    if c not in {"source_file", "raw_attack_label", "family_label", "binary_label", "dataset_id", "record_hash", "stratify_label", "stream_index"}
]

transformer = build_transformer(train_frame, feature_columns)

x_train_base = transformer.fit_transform(train_frame[feature_columns])
x_valid_base = transformer.transform(valid_frame[feature_columns])
x_test_base = transformer.transform(nsl_test_frame[feature_columns])

scaler = StandardScaler()
x_train = scaler.fit_transform(x_train_base).astype(np.float32)
x_valid = scaler.transform(x_valid_base).astype(np.float32)
x_test = scaler.transform(x_test_base).astype(np.float32)

y_train = train_frame["binary_label"].astype(int).values
y_valid = valid_frame["binary_label"].astype(int).values
y_test = nsl_test_frame["binary_label"].astype(int).values

train_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
trained_model = fit_model(x_train, y_train, x_valid, y_valid, device=train_device, config=train_config)

live_model = DenseModel(x_train.shape[1])
live_model.load_state_dict(copy.deepcopy(trained_model.state_dict()))
live_model = live_model.to(torch.device("cpu"))
live_model.eval()

_, full_test_prob, full_test_pred = evaluate_model(
    live_model,
    x_test,
    y_test,
    device=torch.device("cpu"),
    batch_size=train_config["batch_size"]
)

full_test_accuracy = accuracy_score(y_test, full_test_pred)
full_test_balanced_accuracy = balanced_accuracy_score(y_test, full_test_pred)
full_test_precision = precision_score(y_test, full_test_pred, zero_division=0)
full_test_recall = recall_score(y_test, full_test_pred, zero_division=0)
full_test_f1 = f1_score(y_test, full_test_pred, zero_division=0)
full_test_auc = roc_auc_score(y_test, full_test_prob)

live_block_count = min(ledger_config["live_stream_blocks"], len(x_test))
x_live = x_test[:live_block_count]
y_live = y_test[:live_block_count]
stream_index_live = nsl_test_frame["stream_index"].values[:live_block_count]

live_ledger = PermissionedLedger(node_count=ledger_config["live_node_count"])
live_rows = []
payloads_for_benchmark = []

live_wall_start = time.perf_counter()

for i in range(live_block_count):
    inference_start = time.perf_counter()
    with torch.no_grad():
        x_tensor = torch.from_numpy(x_live[i:i + 1]).float()
        logits = live_model(x_tensor)
        attack_probability = torch.sigmoid(logits).cpu().numpy().reshape(-1)[0]
    inference_latency_ms = (time.perf_counter() - inference_start) * 1000.0

    predicted_label_live = int(attack_probability >= 0.5)
    confidence_value = float(max(attack_probability, 1.0 - attack_probability))

    payload = {
        "dataset_id": "nsl_kdd",
        "sample_index": int(stream_index_live[i]),
        "true_label": int(y_live[i]),
        "predicted_label": int(predicted_label_live),
        "attack_probability": round(float(attack_probability), 6),
        "confidence": round(float(confidence_value), 6),
        "model_name": "dense_holdout_detector",
        "logging_mode": "permissioned_hash_chain"
    }

    payloads_for_benchmark.append(copy.deepcopy(payload))

    block, local_append_latency_ms, replicated_commit_latency_ms = live_ledger.commit(payload)
    end_to_end_latency_ms = inference_latency_ms + replicated_commit_latency_ms

    live_rows.append({
        "block_index": int(block["block_index"]),
        "sample_index": int(payload["sample_index"]),
        "true_label": int(payload["true_label"]),
        "predicted_label_live": int(predicted_label_live),
        "predicted_label_on_chain": int(block["predicted_label"]),
        "attack_probability": float(attack_probability),
        "confidence": float(confidence_value),
        "timestamp_utc": block["timestamp_utc"],
        "previous_hash": block["previous_hash"],
        "block_hash": block["block_hash"],
        "previous_hash_prefix": block["previous_hash"][:12],
        "block_hash_prefix": block["block_hash"][:12],
        "local_append_latency_ms": float(local_append_latency_ms),
        "replicated_commit_latency_ms": float(replicated_commit_latency_ms),
        "inference_latency_ms": float(inference_latency_ms),
        "end_to_end_latency_ms": float(end_to_end_latency_ms),
        "block_bytes": int(block["block_bytes"]),
        "node_count": int(ledger_config["live_node_count"]),
        "link_verified": 1
    })

live_wall_elapsed = time.perf_counter() - live_wall_start
live_throughput_blocks_per_sec = float(live_block_count / live_wall_elapsed) if live_wall_elapsed > 0 else np.nan

live_verification = live_ledger.verify_all()
live_chain_valid_all = int(all(item["chain_valid"] == 1 for item in live_verification))
live_chain_valid_primary = int(live_verification[0]["chain_valid"])

live_log_frame = pd.DataFrame(live_rows)
live_log_frame.head(min(ledger_config["sample_export_blocks"], len(live_log_frame))).to_csv(
    output_dir / "phase5_block_log_sample.csv",
    index=False
)

benchmark_rows = []
benchmark_ledgers = {}

for node_count in ledger_config["benchmark_node_counts"]:
    row, ledger_obj = benchmark_ledger(
        payloads=payloads_for_benchmark,
        node_count=node_count,
        throughput_window=ledger_config["throughput_window"]
    )
    benchmark_rows.append(row)
    benchmark_ledgers[node_count] = ledger_obj

benchmark_frame = pd.DataFrame(benchmark_rows).sort_values("node_count").reset_index(drop=True)
benchmark_frame.to_csv(output_dir / "phase5_blockchain_benchmarks.csv", index=False)

single_node_row = benchmark_frame[benchmark_frame["node_count"] == 1].iloc[0]
replicated_row = benchmark_frame[benchmark_frame["node_count"] == ledger_config["live_node_count"]].iloc[0]

buffer_model = io.BytesIO()
torch.save(live_model.state_dict(), buffer_model)
model_artifact_mb = len(buffer_model.getvalue()) / (1024 ** 2)

preprocessing_artifact_mb = len(pickle.dumps({"transformer": transformer, "scaler": scaler})) / (1024 ** 2)
parameter_count = int(sum(p.numel() for p in live_model.parameters()))

mean_block_bytes = float(live_log_frame["block_bytes"].mean()) if len(live_log_frame) > 0 else 0.0
chain_bytes_per_node_live = float(live_log_frame["block_bytes"].sum())
chain_bytes_all_nodes_live = chain_bytes_per_node_live * ledger_config["live_node_count"]

system_rows = [
    {
        "metric_name": "model_parameters",
        "metric_value": parameter_count,
        "unit": "count",
        "notes": "Trainable parameters in the deployed detector"
    },
    {
        "metric_name": "model_artifact_size",
        "metric_value": model_artifact_mb,
        "unit": "MB",
        "notes": "Serialized model state"
    },
    {
        "metric_name": "preprocessing_artifact_size",
        "metric_value": preprocessing_artifact_mb,
        "unit": "MB",
        "notes": "Serialized transformer and scaler bundle"
    },
    {
        "metric_name": "live_blocks_logged",
        "metric_value": live_block_count,
        "unit": "blocks",
        "notes": "Consecutive hold-out predictions committed to the ledger"
    },
    {
        "metric_name": "live_mean_inference_latency",
        "metric_value": float(live_log_frame["inference_latency_ms"].mean()),
        "unit": "ms",
        "notes": "Per-sample model inference on the live stream"
    },
    {
        "metric_name": "live_mean_end_to_end_latency",
        "metric_value": float(live_log_frame["end_to_end_latency_ms"].mean()),
        "unit": "ms",
        "notes": "Inference plus replicated logging"
    },
    {
        "metric_name": "live_throughput",
        "metric_value": live_throughput_blocks_per_sec,
        "unit": "blocks_per_sec",
        "notes": "Observed end-to-end live throughput"
    },
    {
        "metric_name": "mean_block_size",
        "metric_value": mean_block_bytes,
        "unit": "bytes",
        "notes": "Average serialized block size"
    },
    {
        "metric_name": "chain_bytes_per_node_live",
        "metric_value": chain_bytes_per_node_live,
        "unit": "bytes",
        "notes": "Ledger growth for one node over the live sample"
    },
    {
        "metric_name": "chain_bytes_all_nodes_live",
        "metric_value": chain_bytes_all_nodes_live,
        "unit": "bytes",
        "notes": "Replicated ledger growth across all live nodes"
    },
    {
        "metric_name": "single_node_mean_local_append_latency",
        "metric_value": float(single_node_row["mean_local_append_latency_ms"]),
        "unit": "ms",
        "notes": "Logging benchmark with one node"
    },
    {
        "metric_name": "replicated_mean_commit_latency",
        "metric_value": float(replicated_row["mean_replicated_commit_latency_ms"]),
        "unit": "ms",
        "notes": "Logging benchmark with the configured live node count"
    },
    {
        "metric_name": "single_node_peak_throughput",
        "metric_value": float(single_node_row["peak_window_throughput_blocks_per_sec"]),
        "unit": "blocks_per_sec",
        "notes": "Peak rolling-window throughput under single-node logging"
    },
    {
        "metric_name": "replicated_sustained_throughput",
        "metric_value": float(replicated_row["sustained_throughput_blocks_per_sec"]),
        "unit": "blocks_per_sec",
        "notes": "Sustained throughput under replicated logging"
    },
    {
        "metric_name": "replicated_cpu_utilization",
        "metric_value": float(replicated_row["cpu_utilization_percent"]),
        "unit": "percent",
        "notes": "Observed CPU utilization during the replicated benchmark"
    },
    {
        "metric_name": "replicated_peak_rss",
        "metric_value": float(replicated_row["peak_rss_mb"]),
        "unit": "MB",
        "notes": "Observed process memory during the replicated benchmark"
    },
    {
        "metric_name": "primary_chain_valid",
        "metric_value": live_chain_valid_primary,
        "unit": "flag",
        "notes": "Hash linkage check for the primary node"
    },
    {
        "metric_name": "all_nodes_chain_valid",
        "metric_value": live_chain_valid_all,
        "unit": "flag",
        "notes": "Hash linkage check across all live nodes"
    },
    {
        "metric_name": "holdout_accuracy",
        "metric_value": full_test_accuracy,
        "unit": "score",
        "notes": "Full NSL-KDD hold-out accuracy of the deployed detector"
    },
    {
        "metric_name": "holdout_balanced_accuracy",
        "metric_value": full_test_balanced_accuracy,
        "unit": "score",
        "notes": "Full NSL-KDD hold-out balanced accuracy"
    },
    {
        "metric_name": "holdout_precision",
        "metric_value": full_test_precision,
        "unit": "score",
        "notes": "Full NSL-KDD hold-out attack precision"
    },
    {
        "metric_name": "holdout_recall",
        "metric_value": full_test_recall,
        "unit": "score",
        "notes": "Full NSL-KDD hold-out attack recall"
    },
    {
        "metric_name": "holdout_f1",
        "metric_value": full_test_f1,
        "unit": "score",
        "notes": "Full NSL-KDD hold-out attack F1"
    },
    {
        "metric_name": "holdout_roc_auc",
        "metric_value": full_test_auc,
        "unit": "score",
        "notes": "Full NSL-KDD hold-out ROC-AUC"
    }
]

system_frame = pd.DataFrame(system_rows)
system_frame.to_csv(output_dir / "phase5_system_footprint.csv", index=False)

node_count_for_cost = ledger_config["live_node_count"]
provisioned_vcpu_per_node = max(1.0, math.ceil((float(single_node_row["cpu_utilization_percent"]) / 100.0 * max(os.cpu_count(), 1)) * 2.0) / 2.0)
provisioned_memory_gb_per_node = max(1.0, math.ceil((float(single_node_row["peak_rss_mb"]) / 1024.0) * 2.0) / 2.0)

artifact_storage_gb = (model_artifact_mb + preprocessing_artifact_mb) / 1024.0
seconds_per_month = 30.0 * 24.0 * 3600.0

observed_blocks_per_month = live_throughput_blocks_per_sec * seconds_per_month
replicated_capacity_blocks_per_month = float(replicated_row["sustained_throughput_blocks_per_sec"]) * seconds_per_month

monthly_cost_rows = []

for scenario_name, blocks_per_month in [
    ("observed_live_rate", observed_blocks_per_month),
    ("replicated_benchmark_capacity", replicated_capacity_blocks_per_month)
]:
    ledger_storage_gb = (blocks_per_month * mean_block_bytes * node_count_for_cost) / (1024.0 ** 3)

    cpu_hours = provisioned_vcpu_per_node * node_count_for_cost * cost_config["hours_per_month"]
    memory_gb_hours = provisioned_memory_gb_per_node * node_count_for_cost * cost_config["hours_per_month"]

    cpu_cost = cpu_hours * cost_config["vcpu_hour_rate_usd"]
    memory_cost = memory_gb_hours * cost_config["memory_gb_hour_rate_usd"]
    ledger_storage_cost = ledger_storage_gb * cost_config["storage_gb_month_rate_usd"]
    artifact_storage_cost = artifact_storage_gb * cost_config["artifact_storage_gb_month_rate_usd"]

    monthly_cost_rows.append({
        "scenario_name": scenario_name,
        "component_name": "provisioned_vcpu_hours",
        "quantity": cpu_hours,
        "unit": "vcpu_hours_per_month",
        "unit_rate_usd": cost_config["vcpu_hour_rate_usd"],
        "monthly_cost_usd": cpu_cost,
        "notes": f"{node_count_for_cost} nodes at {provisioned_vcpu_per_node:.1f} vCPU per node"
    })
    monthly_cost_rows.append({
        "scenario_name": scenario_name,
        "component_name": "provisioned_memory_gb_hours",
        "quantity": memory_gb_hours,
        "unit": "gb_hours_per_month",
        "unit_rate_usd": cost_config["memory_gb_hour_rate_usd"],
        "monthly_cost_usd": memory_cost,
        "notes": f"{node_count_for_cost} nodes at {provisioned_memory_gb_per_node:.1f} GB per node"
    })
    monthly_cost_rows.append({
        "scenario_name": scenario_name,
        "component_name": "ledger_storage",
        "quantity": ledger_storage_gb,
        "unit": "gb_month",
        "unit_rate_usd": cost_config["storage_gb_month_rate_usd"],
        "monthly_cost_usd": ledger_storage_cost,
        "notes": "Replicated ledger footprint estimated from measured block size"
    })
    monthly_cost_rows.append({
        "scenario_name": scenario_name,
        "component_name": "artifact_storage",
        "quantity": artifact_storage_gb,
        "unit": "gb_month",
        "unit_rate_usd": cost_config["artifact_storage_gb_month_rate_usd"],
        "monthly_cost_usd": artifact_storage_cost,
        "notes": "Model and preprocessing bundle storage"
    })
    monthly_cost_rows.append({
        "scenario_name": scenario_name,
        "component_name": "estimated_total",
        "quantity": np.nan,
        "unit": "usd_per_month",
        "unit_rate_usd": np.nan,
        "monthly_cost_usd": cpu_cost + memory_cost + ledger_storage_cost + artifact_storage_cost,
        "notes": "Editable worksheet total based on the stated assumptions"
    })

monthly_cost_frame = pd.DataFrame(monthly_cost_rows)
monthly_cost_frame.to_csv(output_dir / "phase5_monthly_cost.csv", index=False)

chain_plot_blocks = min(6, len(live_ledger.nodes[0].blocks))
blocks_for_plot = live_ledger.nodes[0].blocks[:chain_plot_blocks]

plt.figure(figsize=(14, 3.8))
ax = plt.gca()
ax.set_xlim(-0.5, chain_plot_blocks - 0.5)
ax.set_ylim(0.0, 1.0)
ax.axis("off")

for idx, block in enumerate(blocks_for_plot):
    box_text = (
        f"idx={block['block_index']}\n"
        f"label={block['predicted_label']}\n"
        f"conf={block['confidence']:.3f}\n"
        f"prev={block['previous_hash'][:10]}\n"
        f"hash={block['block_hash'][:10]}"
    )
    ax.text(
        idx,
        0.55,
        box_text,
        ha="center",
        va="center",
        fontsize=9,
        bbox=dict(boxstyle="round,pad=0.4", linewidth=1.0)
    )
    if idx < chain_plot_blocks - 1:
        ax.annotate(
            "",
            xy=(idx + 0.6, 0.55),
            xytext=(idx + 0.4, 0.55),
            arrowprops=dict(arrowstyle="->", lw=1.5)
        )

plt.tight_layout()
plt.savefig(output_dir / "phase5_hash_chain_flow.png", dpi=300, bbox_inches="tight")
plt.close()

plt.figure(figsize=(9, 5))
plt.plot(live_log_frame["block_index"], live_log_frame["confidence"])
plt.xlabel("Block index")
plt.ylabel("Confidence")
plt.tight_layout()
plt.savefig(output_dir / "phase5_live_confidence.png", dpi=300, bbox_inches="tight")
plt.close()

live_label_counts = live_log_frame["predicted_label_live"].value_counts().reindex([0, 1], fill_value=0)
chain_label_counts = live_log_frame["predicted_label_on_chain"].value_counts().reindex([0, 1], fill_value=0)

positions = np.arange(2)
width = 0.35

plt.figure(figsize=(7, 5))
plt.bar(positions - width / 2, live_label_counts.values, width=width, label="Live run")
plt.bar(positions + width / 2, chain_label_counts.values, width=width, label="On-chain")
plt.xticks(positions, ["Benign", "Attack"])
plt.ylabel("Count")
plt.legend()
plt.tight_layout()
plt.savefig(output_dir / "phase5_label_distribution.png", dpi=300, bbox_inches="tight")
plt.close()

In [ ]:
from pathlib import Path
import copy
import json
import random
import re
import warnings

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, average_precision_score, balanced_accuracy_score, confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import StratifiedGroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder, StandardScaler

warnings.filterwarnings("ignore")

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

output_dir = Path("/kaggle/working/results/phase6")
output_dir.mkdir(parents=True, exist_ok=True)

data_sources = {
    "nsl_kdd": {
        "train": [
            "/kaggle/input/nsl-kdd/KDDTrain+.txt"
        ],
        "test": [
            "/kaggle/input/nsl-kdd/KDDTest+.txt"
        ]
    },
    "cicids2017": [
        "/kaggle/input/cicids2017"
    ],
    "unsw_nb15": [
        "/kaggle/input/unsw-nb15"
    ]
}

split_config = {
    "train_fraction": 0.70,
    "valid_fraction": 0.15,
    "test_fraction": 0.15,
    "random_state": 42
}

train_config = {
    "epochs": 35,
    "batch_size": 4096,
    "learning_rate": 1e-3,
    "weight_decay": 1e-4,
    "patience": 7
}

robustness_config = {
    "gaussian_sigma": 0.05
}

nsl_columns = [
    "duration", "protocol_type", "service", "flag", "src_bytes", "dst_bytes", "land",
    "wrong_fragment", "urgent", "hot", "num_failed_logins", "logged_in", "num_compromised",
    "root_shell", "su_attempted", "num_root", "num_file_creations", "num_shells",
    "num_access_files", "num_outbound_cmds", "is_host_login", "is_guest_login", "count",
    "srv_count", "serror_rate", "srv_serror_rate", "rerror_rate", "srv_rerror_rate",
    "same_srv_rate", "diff_srv_rate", "srv_diff_host_rate", "dst_host_count",
    "dst_host_srv_count", "dst_host_same_srv_rate", "dst_host_diff_srv_rate",
    "dst_host_same_src_port_rate", "dst_host_srv_diff_host_rate", "dst_host_serror_rate",
    "dst_host_srv_serror_rate", "dst_host_rerror_rate", "dst_host_srv_rerror_rate",
    "raw_label", "difficulty"
]

nsl_family_map = {
    "normal": "normal",
    "back": "dos",
    "land": "dos",
    "neptune": "dos",
    "pod": "dos",
    "smurf": "dos",
    "teardrop": "dos",
    "mailbomb": "dos",
    "apache2": "dos",
    "processtable": "dos",
    "udpstorm": "dos",
    "ipsweep": "probe",
    "nmap": "probe",
    "portsweep": "probe",
    "satan": "probe",
    "mscan": "probe",
    "saint": "probe",
    "ftp_write": "r2l",
    "guess_passwd": "r2l",
    "imap": "r2l",
    "multihop": "r2l",
    "phf": "r2l",
    "spy": "r2l",
    "warezclient": "r2l",
    "warezmaster": "r2l",
    "sendmail": "r2l",
    "named": "r2l",
    "snmpgetattack": "r2l",
    "snmpguess": "r2l",
    "xlock": "r2l",
    "xsnoop": "r2l",
    "worm": "r2l",
    "buffer_overflow": "u2r",
    "loadmodule": "u2r",
    "perl": "u2r",
    "rootkit": "u2r",
    "httptunnel": "u2r",
    "ps": "u2r",
    "sqlattack": "u2r",
    "xterm": "u2r"
}

attack_label_candidates = [
    "label", "labels", "class", "target", "attack", "attack_type", "attack_cat",
    "category", "traffic_label", "raw_label", "rawlabel"
]

family_label_candidates = [
    "attack_family", "family", "attack_cat", "attack_category", "category_group"
]

drop_candidates = {
    "flow_id", "flowid", "src_ip", "dst_ip", "srcip", "dstip", "timestamp", "time", "id",
    "index", "idx", "session_id", "record_id"
}


def normalize_name(value):
    value = str(value).strip().lower()
    value = re.sub(r"[\s\-/]+", "_", value)
    value = re.sub(r"[^a-z0-9_]", "", value)
    value = re.sub(r"_+", "_", value).strip("_")
    return value


def normalize_text(value):
    if pd.isna(value):
        return ""
    value = str(value).strip().lower()
    value = value.replace("\x00", "")
    value = re.sub(r"\s+", " ", value)
    return value


def resolve_files(entries):
    files = []
    for entry in entries:
        path_obj = Path(entry)
        if path_obj.is_dir():
            files.extend(sorted([p for p in path_obj.rglob("*") if p.suffix.lower() in {".csv", ".txt", ".data", ".parquet"}]))
        elif path_obj.exists():
            files.append(path_obj)
    return files


def read_single_file(path_obj):
    suffix = path_obj.suffix.lower()
    if suffix == ".parquet":
        return pd.read_parquet(path_obj)
    try:
        frame = pd.read_csv(path_obj, low_memory=False)
        if frame.shape[1] > 1:
            return frame
    except Exception:
        pass
    try:
        frame = pd.read_csv(path_obj, header=None, low_memory=False)
        if frame.shape[1] > 1:
            return frame
    except Exception:
        pass
    try:
        return pd.read_csv(path_obj, sep=None, engine="python", low_memory=False)
    except Exception:
        return pd.read_csv(path_obj, sep=r"\s+", engine="python", low_memory=False)


def standardize_columns(frame):
    cols = [normalize_name(c) for c in frame.columns]
    used = {}
    final_cols = []
    for col in cols:
        if col not in used:
            used[col] = 0
            final_cols.append(col)
        else:
            used[col] += 1
            final_cols.append(f"{col}_{used[col]}")
    frame.columns = final_cols
    return frame


def assign_known_schema(dataset_key, frame):
    if dataset_key == "nsl_kdd":
        if frame.shape[1] == len(nsl_columns):
            frame.columns = nsl_columns
        elif frame.shape[1] == len(nsl_columns) - 1:
            frame.columns = nsl_columns[:-1]
            frame["difficulty"] = 0
    return frame


def load_dataset(dataset_key, entries):
    parts = []
    for path_obj in resolve_files(entries):
        frame = read_single_file(path_obj)
        frame = assign_known_schema(dataset_key, frame)
        frame = standardize_columns(frame)
        frame["source_file"] = path_obj.name
        parts.append(frame)
    if not parts:
        raise FileNotFoundError(dataset_key)
    frame = pd.concat(parts, axis=0, ignore_index=True)
    frame = standardize_columns(frame)
    return frame


def select_first_present(columns, candidates):
    lookup = {normalize_name(c): c for c in columns}
    for candidate in candidates:
        key = normalize_name(candidate)
        if key in lookup:
            return lookup[key]
    return None


def derive_label_columns(dataset_key, frame):
    label_col = select_first_present(frame.columns, attack_label_candidates)
    family_col = select_first_present(frame.columns, family_label_candidates)

    if dataset_key == "nsl_kdd":
        if "raw_label" in frame.columns:
            label_col = "raw_label"
        family_col = None

    if dataset_key == "unsw_nb15":
        if "attack_cat" in frame.columns:
            family_col = "attack_cat"
        if "label" in frame.columns:
            label_col = "label"

    if dataset_key == "cicids2017":
        if "label" in frame.columns:
            label_col = "label"

    if label_col is None:
        raise ValueError(dataset_key)
    return label_col, family_col


def coerce_basic_types(frame):
    frame = frame.copy()
    frame = frame.replace([np.inf, -np.inf], np.nan)
    for col in frame.columns:
        if frame[col].dtype == object:
            converted = pd.to_numeric(frame[col], errors="coerce")
            if converted.notna().mean() >= 0.90:
                frame[col] = converted
    missing_only = [c for c in frame.columns if frame[c].isna().all()]
    if missing_only:
        frame = frame.drop(columns=missing_only)
    return frame


def drop_redundant_columns(frame, protected):
    drop_cols = []
    for col in frame.columns:
        if col in protected:
            continue
        if normalize_name(col) in drop_candidates:
            drop_cols.append(col)
            continue
        if frame[col].nunique(dropna=False) <= 1:
            drop_cols.append(col)
    if drop_cols:
        frame = frame.drop(columns=drop_cols)
    return frame


def nsl_family_from_label(raw_value):
    raw_value = normalize_text(raw_value).rstrip(".")
    return nsl_family_map.get(raw_value, "other_attack")


def cicids_family_from_label(raw_value):
    value = normalize_text(raw_value)
    if value in {"benign", "normal"}:
        return "normal"
    if "ddos" in value:
        return "ddos"
    if value.startswith("dos") or "hulk" in value or "goldeneye" in value or "slowhttptest" in value or "slowloris" in value:
        return "dos"
    if "portscan" in value or "port_scan" in value:
        return "portscan"
    if "bot" in value:
        return "botnet"
    if "web attack" in value or "sql injection" in value or "xss" in value or "brute force" in value:
        return "web_attack"
    if "ftp-patator" in value or "ssh-patator" in value:
        return "brute_force"
    if "infiltration" in value:
        return "infiltration"
    if "heartbleed" in value:
        return "heartbleed"
    return normalize_name(value) if value else "unknown"


def unsw_family_from_values(raw_label_value, family_value):
    family_text = normalize_text(family_value)
    if family_text:
        if family_text in {"normal", "benign"}:
            return "normal"
        return normalize_name(family_text)
    raw_text = normalize_text(raw_label_value)
    if raw_text in {"0", "normal", "benign"}:
        return "normal"
    if raw_text in {"1", "attack", "malicious"}:
        return "generic_attack"
    return "unknown"


def derive_family(dataset_key, raw_label_value, family_value=None):
    if dataset_key == "nsl_kdd":
        return nsl_family_from_label(raw_label_value)
    if dataset_key == "cicids2017":
        return cicids_family_from_label(raw_label_value)
    if dataset_key == "unsw_nb15":
        return unsw_family_from_values(raw_label_value, family_value)
    family_text = normalize_text(family_value)
    if family_text in {"normal", "benign"}:
        return "normal"
    if family_text:
        return normalize_name(family_text)
    raw_text = normalize_text(raw_label_value)
    if raw_text in {"0", "normal", "benign"}:
        return "normal"
    if raw_text in {"1", "attack", "malicious"}:
        return "generic_attack"
    return normalize_name(raw_text) if raw_text else "unknown"


def stable_record_hash(frame, feature_columns):
    hash_frame = frame[feature_columns].copy()
    for col in hash_frame.columns:
        if pd.api.types.is_numeric_dtype(hash_frame[col]):
            hash_frame[col] = hash_frame[col].astype("float64").round(8)
        else:
            hash_frame[col] = hash_frame[col].astype(str).str.strip().str.lower()
    return pd.util.hash_pandas_object(hash_frame, index=False).astype(str)


def choose_feature_columns(frame, protected_columns):
    return [c for c in frame.columns if c not in protected_columns]


def make_split_column(frame, stratify_col, group_col, config):
    splitter_outer = StratifiedGroupShuffleSplit(
        n_splits=1,
        test_size=config["test_fraction"],
        random_state=config["random_state"]
    )
    outer_train_idx, test_idx = next(splitter_outer.split(frame, frame[stratify_col], groups=frame[group_col]))

    remainder = frame.iloc[outer_train_idx].copy()
    remainder_y = remainder[stratify_col]
    remainder_groups = remainder[group_col]

    inner_valid_fraction = config["valid_fraction"] / (config["train_fraction"] + config["valid_fraction"])
    splitter_inner = StratifiedGroupShuffleSplit(
        n_splits=1,
        test_size=inner_valid_fraction,
        random_state=config["random_state"]
    )
    train_idx_local, valid_idx_local = next(splitter_inner.split(remainder, remainder_y, groups=remainder_groups))

    frame = frame.copy()
    frame["split_name"] = "train"
    frame.loc[frame.index[test_idx], "split_name"] = "test"
    frame.loc[remainder.index[valid_idx_local], "split_name"] = "valid"
    frame.loc[remainder.index[train_idx_local], "split_name"] = "train"
    return frame


def make_train_valid_split(frame, valid_fraction, random_state):
    frame = frame.copy()
    splitter = StratifiedGroupShuffleSplit(
        n_splits=1,
        test_size=valid_fraction,
        random_state=random_state
    )
    train_idx, valid_idx = next(splitter.split(frame, frame["stratify_label"], groups=frame["record_hash"]))
    train_frame = frame.iloc[train_idx].copy().reset_index(drop=True)
    valid_frame = frame.iloc[valid_idx].copy().reset_index(drop=True)
    return train_frame, valid_frame


def build_transformer(frame, feature_columns):
    numeric_columns = [c for c in feature_columns if pd.api.types.is_numeric_dtype(frame[c])]
    categorical_columns = [c for c in feature_columns if c not in numeric_columns]

    numeric_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median"))
    ])

    categorical_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1))
    ])

    transformer = ColumnTransformer(
        transformers=[
            ("num", numeric_pipe, numeric_columns),
            ("cat", categorical_pipe, categorical_columns)
        ],
        remainder="drop"
    )

    return transformer


class DenseModel(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.body = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.20),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.15),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        return self.body(x)


def evaluate_model(model, x_data, y_data, device, batch_size):
    criterion = nn.BCEWithLogitsLoss()
    losses = []
    probs_all = []

    model.eval()
    with torch.no_grad():
        for start in range(0, len(x_data), batch_size):
            xb = torch.from_numpy(x_data[start:start + batch_size]).float().to(device)
            yb = torch.from_numpy(y_data[start:start + batch_size]).float().view(-1, 1).to(device)
            logits = model(xb)
            loss = criterion(logits, yb)
            probs = torch.sigmoid(logits).detach().cpu().numpy().reshape(-1)
            losses.append(loss.item() * len(xb))
            probs_all.append(probs)

    probs_all = np.concatenate(probs_all, axis=0)
    preds = (probs_all >= 0.5).astype(int)
    avg_loss = float(np.sum(losses) / len(x_data))
    return avg_loss, probs_all, preds


def fit_model(x_train, y_train, x_valid, y_valid, device, config):
    model = DenseModel(x_train.shape[1]).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=config["learning_rate"], weight_decay=config["weight_decay"])
    criterion = nn.BCEWithLogitsLoss()

    best_state = copy.deepcopy(model.state_dict())
    best_loss = np.inf
    wait = 0
    indices = np.arange(len(x_train))

    for _ in range(config["epochs"]):
        np.random.shuffle(indices)
        model.train()

        for start in range(0, len(indices), config["batch_size"]):
            idx = indices[start:start + config["batch_size"]]
            xb = torch.from_numpy(x_train[idx]).float().to(device)
            yb = torch.from_numpy(y_train[idx]).float().view(-1, 1).to(device)

            optimizer.zero_grad(set_to_none=True)
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()

        valid_loss, _, _ = evaluate_model(model, x_valid, y_valid, device=device, batch_size=config["batch_size"])

        if valid_loss < best_loss - 1e-6:
            best_loss = valid_loss
            best_state = copy.deepcopy(model.state_dict())
            wait = 0
        else:
            wait += 1
            if wait >= config["patience"]:
                break

    model.load_state_dict(best_state)
    model.eval()
    return model


def gaussian_corruption(x_data, sigma, clip_min, clip_max, seed=42):
    rng = np.random.default_rng(seed)
    noise = rng.normal(0.0, sigma, size=x_data.shape).astype(np.float32)
    corrupted = x_data + noise
    corrupted = np.clip(corrupted, clip_min, clip_max)
    return corrupted.astype(np.float32)


def metric_bundle(y_true, y_prob, threshold=0.5):
    y_pred = (y_prob >= threshold).astype(int)
    acc = accuracy_score(y_true, y_pred)
    bacc = balanced_accuracy_score(y_true, y_pred)
    pre = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    f1v = f1_score(y_true, y_pred, zero_division=0)
    macro_pre = precision_score(y_true, y_pred, average="macro", zero_division=0)
    macro_rec = recall_score(y_true, y_pred, average="macro", zero_division=0)
    macro_f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
    try:
        auc = roc_auc_score(y_true, y_prob)
    except Exception:
        auc = np.nan
    try:
        ap = average_precision_score(y_true, y_prob)
    except Exception:
        ap = np.nan
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    fpr = fp / (fp + tn) if (fp + tn) > 0 else np.nan
    return {
        "accuracy": acc,
        "balanced_accuracy": bacc,
        "precision": pre,
        "recall": rec,
        "f1": f1v,
        "macro_precision": macro_pre,
        "macro_recall": macro_rec,
        "macro_f1": macro_f1,
        "roc_auc": auc,
        "pr_auc": ap,
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
        "false_positive_rate": fpr,
        "misclassifications": int(fp + fn)
    }


def prepare_frame(dataset_key, frame):
    frame = standardize_columns(frame)
    frame = coerce_basic_types(frame)
    label_col, family_col = derive_label_columns(dataset_key, frame)

    work_frame = frame.copy()
    work_frame["raw_attack_label"] = work_frame[label_col].astype(str).map(normalize_text)

    if family_col is not None and family_col in work_frame.columns:
        family_values = work_frame[family_col]
    else:
        family_values = pd.Series([""] * len(work_frame), index=work_frame.index)

    work_frame["family_label"] = [
        derive_family(dataset_key, raw_val, fam_val)
        for raw_val, fam_val in zip(work_frame["raw_attack_label"], family_values)
    ]
    work_frame["binary_label"] = (work_frame["family_label"] != "normal").astype(int)
    work_frame["dataset_id"] = dataset_key

    protected = {"source_file", "raw_attack_label", "family_label", "binary_label", "dataset_id"}
    work_frame = drop_redundant_columns(work_frame, protected)
    return work_frame


def finalize_frame(frame):
    work_frame = frame.copy()
    feature_columns = choose_feature_columns(work_frame, {"source_file", "raw_attack_label", "family_label", "binary_label", "dataset_id"})
    work_frame["record_hash"] = stable_record_hash(work_frame, feature_columns)
    work_frame = work_frame.drop_duplicates(subset=["record_hash"]).reset_index(drop=True)
    feature_columns = choose_feature_columns(work_frame, {"source_file", "raw_attack_label", "family_label", "binary_label", "dataset_id", "record_hash"})
    stratify_source = work_frame["family_label"].copy()
    rare_families = stratify_source.value_counts()
    rare_families = set(rare_families[rare_families < 3].index.tolist())
    stratify_source = stratify_source.apply(lambda x: "rare_attack" if x in rare_families and x != "normal" else x)
    work_frame["stratify_label"] = stratify_source.astype(str)
    return work_frame


def run_protocol(dataset_key):
    if dataset_key == "nsl_kdd":
        train_raw = load_dataset("nsl_kdd", data_sources["nsl_kdd"]["train"])
        test_raw = load_dataset("nsl_kdd", data_sources["nsl_kdd"]["test"])

        train_frame = finalize_frame(prepare_frame("nsl_kdd", train_raw))
        test_frame = finalize_frame(prepare_frame("nsl_kdd", test_raw))

        shared_columns = sorted(set(train_frame.columns).intersection(set(test_frame.columns)))
        train_frame = train_frame[shared_columns].copy()
        test_frame = test_frame[shared_columns].copy()

        train_split, valid_split = make_train_valid_split(
            train_frame,
            valid_fraction=split_config["valid_fraction"],
            random_state=split_config["random_state"]
        )
        test_split = test_frame.copy()
        train_split["split_name"] = "train"
        valid_split["split_name"] = "valid"
        test_split["split_name"] = "test"
    else:
        raw_frame = load_dataset(dataset_key, data_sources[dataset_key])
        full_frame = finalize_frame(prepare_frame(dataset_key, raw_frame))
        full_frame = make_split_column(full_frame, "stratify_label", "record_hash", split_config)
        train_split = full_frame[full_frame["split_name"] == "train"].copy().reset_index(drop=True)
        valid_split = full_frame[full_frame["split_name"] == "valid"].copy().reset_index(drop=True)
        test_split = full_frame[full_frame["split_name"] == "test"].copy().reset_index(drop=True)

    excluded = {"source_file", "raw_attack_label", "family_label", "binary_label", "dataset_id", "record_hash", "stratify_label", "split_name"}
    feature_columns = [c for c in train_split.columns if c not in excluded]
    feature_columns = [c for c in feature_columns if c in valid_split.columns and c in test_split.columns]

    transformer = build_transformer(train_split, feature_columns)

    x_train_base = transformer.fit_transform(train_split[feature_columns])
    x_valid_base = transformer.transform(valid_split[feature_columns])
    x_test_base = transformer.transform(test_split[feature_columns])

    scaler = StandardScaler()
    x_train = scaler.fit_transform(x_train_base).astype(np.float32)
    x_valid = scaler.transform(x_valid_base).astype(np.float32)
    x_test = scaler.transform(x_test_base).astype(np.float32)

    y_train = train_split["binary_label"].astype(int).values
    y_valid = valid_split["binary_label"].astype(int).values
    y_test = test_split["binary_label"].astype(int).values

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = fit_model(x_train, y_train, x_valid, y_valid, device=device, config=train_config)

    _, clean_prob, clean_pred = evaluate_model(model, x_test, y_test, device=device, batch_size=train_config["batch_size"])

    clip_min = x_train.min(axis=0).astype(np.float32)
    clip_max = x_train.max(axis=0).astype(np.float32)
    x_test_gaussian = gaussian_corruption(
        x_test,
        sigma=robustness_config["gaussian_sigma"],
        clip_min=clip_min,
        clip_max=clip_max,
        seed=42
    )
    _, gaussian_prob, gaussian_pred = evaluate_model(model, x_test_gaussian, y_test, device=device, batch_size=train_config["batch_size"])

    clean_metrics = metric_bundle(y_test, clean_prob)
    gaussian_metrics = metric_bundle(y_test, gaussian_prob)

    return {
        "dataset_id": dataset_key,
        "feature_count": int(len(feature_columns)),
        "train_count": int(len(train_split)),
        "valid_count": int(len(valid_split)),
        "test_count": int(len(test_split)),
        "class_benign_test": int((y_test == 0).sum()),
        "class_attack_test": int((y_test == 1).sum()),
        "clean_metrics": clean_metrics,
        "gaussian_metrics": gaussian_metrics,
        "clean_confusion": confusion_matrix(y_test, clean_pred, labels=[0, 1]),
        "gaussian_confusion": confusion_matrix(y_test, gaussian_pred, labels=[0, 1])
    }


results = {}
for dataset_key in ["nsl_kdd", "cicids2017", "unsw_nb15"]:
    results[dataset_key] = run_protocol(dataset_key)

protocol_rows = []

for dataset_key in ["nsl_kdd", "cicids2017", "unsw_nb15"]:
    protocol_rows.append({
        "section": "dataset_scope",
        "dataset_id": dataset_key,
        "item_name": "label_space",
        "item_value": "binary_intrusion_detection",
        "details": "labels harmonized to benign versus attack with family-level taxonomy retained"
    })
    protocol_rows.append({
        "section": "dataset_scope",
        "dataset_id": dataset_key,
        "item_name": "duplicate_policy",
        "item_value": "drop_exact_feature_hash_duplicates_before_split",
        "details": "record_hash generated from cleaned features"
    })
    protocol_rows.append({
        "section": "dataset_scope",
        "dataset_id": dataset_key,
        "item_name": "preprocessing",
        "item_value": "median_impute_numeric_and_encode_categorical_then_standardize",
        "details": "same leakage-controlled preprocessing used across datasets"
    })
    protocol_rows.append({
        "section": "dataset_scope",
        "dataset_id": dataset_key,
        "item_name": "decision_threshold",
        "item_value": "0.5",
        "details": "fixed threshold for the main detector"
    })

protocol_rows.extend([
    {
        "section": "split_policy",
        "dataset_id": "all",
        "item_name": "cross_dataset_split",
        "item_value": f"{split_config['train_fraction']:.2f}/{split_config['valid_fraction']:.2f}/{split_config['test_fraction']:.2f}",
        "details": "used for CICIDS2017 and UNSW-NB15 with grouped stratified splitting"
    },
    {
        "section": "split_policy",
        "dataset_id": "nsl_kdd",
        "item_name": "holdout_policy",
        "item_value": "official_train_test_plus_validation_from_train",
        "details": "official NSL-KDD test retained as the final hold-out"
    },
    {
        "section": "robustness_setting",
        "dataset_id": "all",
        "item_name": "gaussian_sigma",
        "item_value": f"{robustness_config['gaussian_sigma']:.4f}",
        "details": "applied after training-time scaling with feature-wise clipping to observed train ranges"
    },
    {
        "section": "primary_model",
        "dataset_id": "all",
        "item_name": "model_family",
        "item_value": "dense_neural_binary_classifier",
        "details": f"epochs={train_config['epochs']}, batch_size={train_config['batch_size']}, learning_rate={train_config['learning_rate']}, weight_decay={train_config['weight_decay']}, patience={train_config['patience']}"
    },
    {
        "section": "baseline_protocol",
        "dataset_id": "all",
        "item_name": "baseline_1",
        "item_value": "logistic_regression",
        "details": "same splits, same features, same preprocessing, threshold 0.5 where applicable"
    },
    {
        "section": "baseline_protocol",
        "dataset_id": "all",
        "item_name": "baseline_2",
        "item_value": "random_forest",
        "details": "same splits, same binary target, same evaluation metrics"
    },
    {
        "section": "baseline_protocol",
        "dataset_id": "all",
        "item_name": "baseline_3",
        "item_value": "hist_gradient_boosting",
        "details": "same splits, same metrics, no per-dataset retuning beyond the frozen protocol"
    },
    {
        "section": "reporting_rule",
        "dataset_id": "all",
        "item_name": "reported_metrics",
        "item_value": "accuracy_balanced_accuracy_precision_recall_f1_macro_f1_roc_auc_pr_auc_fpr_tp_tn_fp_fn",
        "details": "all models and datasets must report the same family of metrics"
    }
])

protocol_frame = pd.DataFrame(protocol_rows)
protocol_frame.to_csv(output_dir / "phase6_fair_comparison_protocol.csv", index=False)

summary_rows = []

for dataset_key in ["nsl_kdd", "cicids2017", "unsw_nb15"]:
    for condition_name, metrics_key in [("clean", "clean_metrics"), ("gaussian", "gaussian_metrics")]:
        metric_data = results[dataset_key][metrics_key]
        summary_rows.append({
            "section": "benchmark_summary",
            "dataset_id": dataset_key,
            "condition": condition_name,
            "sample_count": results[dataset_key]["test_count"],
            "benign_count": results[dataset_key]["class_benign_test"],
            "attack_count": results[dataset_key]["class_attack_test"],
            "feature_count": results[dataset_key]["feature_count"],
            "train_count": results[dataset_key]["train_count"],
            "valid_count": results[dataset_key]["valid_count"],
            "test_count": results[dataset_key]["test_count"],
            "accuracy": metric_data["accuracy"],
            "balanced_accuracy": metric_data["balanced_accuracy"],
            "precision": metric_data["precision"],
            "recall": metric_data["recall"],
            "f1": metric_data["f1"],
            "macro_precision": metric_data["macro_precision"],
            "macro_recall": metric_data["macro_recall"],
            "macro_f1": metric_data["macro_f1"],
            "roc_auc": metric_data["roc_auc"],
            "pr_auc": metric_data["pr_auc"],
            "false_positive_rate": metric_data["false_positive_rate"],
            "tn": metric_data["tn"],
            "fp": metric_data["fp"],
            "fn": metric_data["fn"],
            "tp": metric_data["tp"],
            "misclassifications": metric_data["misclassifications"],
            "value_text": ""
        })

positioning_rows = [
    {
        "section": "framework_positioning",
        "dataset_id": "all",
        "condition": "not_applicable",
        "sample_count": np.nan,
        "benign_count": np.nan,
        "attack_count": np.nan,
        "feature_count": np.nan,
        "train_count": np.nan,
        "valid_count": np.nan,
        "test_count": np.nan,
        "accuracy": np.nan,
        "balanced_accuracy": np.nan,
        "precision": np.nan,
        "recall": np.nan,
        "f1": np.nan,
        "macro_precision": np.nan,
        "macro_recall": np.nan,
        "macro_f1": np.nan,
        "roc_auc": np.nan,
        "pr_auc": np.nan,
        "false_positive_rate": np.nan,
        "tn": np.nan,
        "fp": np.nan,
        "fn": np.nan,
        "tp": np.nan,
        "misclassifications": np.nan,
        "value_text": "cross_dataset_validation=3_benchmarks"
    },
    {
        "section": "framework_positioning",
        "dataset_id": "all",
        "condition": "not_applicable",
        "sample_count": np.nan,
        "benign_count": np.nan,
        "attack_count": np.nan,
        "feature_count": np.nan,
        "train_count": np.nan,
        "valid_count": np.nan,
        "test_count": np.nan,
        "accuracy": np.nan,
        "balanced_accuracy": np.nan,
        "precision": np.nan,
        "recall": np.nan,
        "f1": np.nan,
        "macro_precision": np.nan,
        "macro_recall": np.nan,
        "macro_f1": np.nan,
        "roc_auc": np.nan,
        "pr_auc": np.nan,
        "false_positive_rate": np.nan,
        "tn": np.nan,
        "fp": np.nan,
        "fn": np.nan,
        "tp": np.nan,
        "misclassifications": np.nan,
        "value_text": "explainability=global_and_local_shap_available_from_phase2"
    },
    {
        "section": "framework_positioning",
        "dataset_id": "all",
        "condition": "not_applicable",
        "sample_count": np.nan,
        "benign_count": np.nan,
        "attack_count": np.nan,
        "feature_count": np.nan,
        "train_count": np.nan,
        "valid_count": np.nan,
        "test_count": np.nan,
        "accuracy": np.nan,
        "balanced_accuracy": np.nan,
        "precision": np.nan,
        "recall": np.nan,
        "f1": np.nan,
        "macro_precision": np.nan,
        "macro_recall": np.nan,
        "macro_f1": np.nan,
        "roc_auc": np.nan,
        "pr_auc": np.nan,
        "false_positive_rate": np.nan,
        "tn": np.nan,
        "fp": np.nan,
        "fn": np.nan,
        "tp": np.nan,
        "misclassifications": np.nan,
        "value_text": "robustness=clean_gaussian_fgsm_pgd_evidence_available"
    },
    {
        "section": "framework_positioning",
        "dataset_id": "all",
        "condition": "not_applicable",
        "sample_count": np.nan,
        "benign_count": np.nan,
        "attack_count": np.nan,
        "feature_count": np.nan,
        "train_count": np.nan,
        "valid_count": np.nan,
        "test_count": np.nan,
        "accuracy": np.nan,
        "balanced_accuracy": np.nan,
        "precision": np.nan,
        "recall": np.nan,
        "f1": np.nan,
        "macro_precision": np.nan,
        "macro_recall": np.nan,
        "macro_f1": np.nan,
        "roc_auc": np.nan,
        "pr_auc": np.nan,
        "false_positive_rate": np.nan,
        "tn": np.nan,
        "fp": np.nan,
        "fn": np.nan,
        "tp": np.nan,
        "misclassifications": np.nan,
        "value_text": "auditability=permissioned_blockchain_logging_available"
    },
    {
        "section": "framework_positioning",
        "dataset_id": "all",
        "condition": "not_applicable",
        "sample_count": np.nan,
        "benign_count": np.nan,
        "attack_count": np.nan,
        "feature_count": np.nan,
        "train_count": np.nan,
        "valid_count": np.nan,
        "test_count": np.nan,
        "accuracy": np.nan,
        "balanced_accuracy": np.nan,
        "precision": np.nan,
        "recall": np.nan,
        "f1": np.nan,
        "macro_precision": np.nan,
        "macro_recall": np.nan,
        "macro_f1": np.nan,
        "roc_auc": np.nan,
        "pr_auc": np.nan,
        "false_positive_rate": np.nan,
        "tn": np.nan,
        "fp": np.nan,
        "fn": np.nan,
        "tp": np.nan,
        "misclassifications": np.nan,
        "value_text": "deployment_readiness=latency_throughput_cost_profile_available"
    }
]

summary_frame = pd.concat([pd.DataFrame(summary_rows), pd.DataFrame(positioning_rows)], axis=0, ignore_index=True)
summary_frame.to_csv(output_dir / "phase6_final_benchmark_summary.csv", index=False)

fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))

for axis_index, dataset_key in enumerate(["cicids2017", "unsw_nb15"]):
    matrix_values = results[dataset_key]["clean_confusion"]
    axes[axis_index].imshow(matrix_values, aspect="auto")
    axes[axis_index].set_xticks([0, 1])
    axes[axis_index].set_xticklabels(["Predicted Benign", "Predicted Attack"], rotation=15)
    axes[axis_index].set_yticks([0, 1])
    axes[axis_index].set_yticklabels(["Actual Benign", "Actual Attack"])
    axes[axis_index].set_title(dataset_key)
    for i in range(matrix_values.shape[0]):
        for j in range(matrix_values.shape[1]):
            axes[axis_index].text(j, i, str(matrix_values[i, j]), ha="center", va="center")
    fig.colorbar(axes[axis_index].images[0], ax=axes[axis_index], fraction=0.046, pad=0.04)

plt.tight_layout()
plt.savefig(output_dir / "phase6_modern_confusion_matrices.png", dpi=300, bbox_inches="tight")
plt.close()

handoff_lines = []

handoff_lines.append("Phase 6 packaging completed with the fixed cross-dataset protocol and manuscript-ready artifacts.")
handoff_lines.append("")
handoff_lines.append("Saved files:")
handoff_lines.append(str(output_dir / "phase6_modern_confusion_matrices.png"))
handoff_lines.append(str(output_dir / "phase6_fair_comparison_protocol.csv"))
handoff_lines.append(str(output_dir / "phase6_final_benchmark_summary.csv"))
handoff_lines.append(str(output_dir / "phase6_results_handoff.txt"))
handoff_lines.append("")
handoff_lines.append("Core result summary:")
for dataset_key in ["nsl_kdd", "cicids2017", "unsw_nb15"]:
    clean_metrics = results[dataset_key]["clean_metrics"]
    gaussian_metrics = results[dataset_key]["gaussian_metrics"]
    handoff_lines.append(
        f"{dataset_key}: clean accuracy {clean_metrics['accuracy']:.4f}, clean macro_f1 {clean_metrics['macro_f1']:.4f}, clean roc_auc {clean_metrics['roc_auc']:.4f}, false_positive_rate {clean_metrics['false_positive_rate']:.4f}, misclassifications {clean_metrics['misclassifications']}"
    )
    handoff_lines.append(
        f"{dataset_key}: gaussian accuracy {gaussian_metrics['accuracy']:.4f}, gaussian macro_f1 {gaussian_metrics['macro_f1']:.4f}, gaussian roc_auc {gaussian_metrics['roc_auc']:.4f}, gaussian false_positive_rate {gaussian_metrics['false_positive_rate']:.4f}"
    )
handoff_lines.append("")
handoff_lines.append("Protocol freeze summary:")
handoff_lines.append("The same binary label harmonization, duplicate filtering, grouped leakage-controlled splitting, preprocessing, thresholding, and reporting metrics were fixed across all benchmarks.")
handoff_lines.append("The protocol table also freezes the comparison settings for logistic regression, random forest, and hist gradient boosting baselines under the same data policy.")
handoff_lines.append("")
handoff_lines.append("Summary file structure:")
handoff_lines.append("The summary csv includes benchmark_summary rows for clean and gaussian conditions on all datasets and framework_positioning rows for final manuscript positioning.")
handoff_lines.append("")
handoff_lines.append("Confusion-matrix packaging note:")
handoff_lines.append("The modern-dataset confusion matrix figure contains one panel for CICIDS2017 and one panel for UNSW-NB15 using the clean hold-out predictions from the fixed protocol.")

with open(output_dir / "phase6_results_handoff.txt", "w", encoding="utf-8") as handle:
    handle.write("\n".join(handoff_lines))